In [ ]:
%run ./0___Reference.ipynb
%run ./0___Reference___Functions.ipynb
%run ./0___Reference___Plots.ipynb

In [ ]:
import sys
import plotly
import plotly.graph_objs as go
import plotly.offline as pyo

from scipy.optimize import curve_fit
from scipy.integrate import quad

import matplotlib.colors as colors

In [ ]:
if not os.path.exists(plots_path_0): os.makedirs(plots_path_0)

In [ ]:
analysis_path = "../Analysis___General_Data/"
if not os.path.exists(analysis_path): os.makedirs(analysis_path)

---

In [ ]:
del all_parameters, file, ap, ALL_cores_no___1_DTFE_KDTree, ALL_cores_no___1_DTFE_Density_Loop, ALL_cores_no___1_NGP, ALL_cores_no___2_Smoothing, ALL_cores_no___3_Layer,ALL_cores_no___4_Origins, ALL_cores_no___5_UOD_vals_and_lvls, ALL_cores_no___6_Levels_isolated_pairs, ALL_cores_no___7_Finder, edge_cut, Density_uod, txt_ZONE, cdf_log, l_grid, i0, size, lvl, z, zPaths_no, sigma_ps, d_xyz_sgm; gc.collect()

---

Note: Remember that for the Finder fiels we removed the voids indices but never reordered the remaining ones so there would be no holes in the ordering: that is for both the under and over density ones.

---

Each section is self-contained: that is, no previous parameters are required, so we always re-define the parameters at the start of each section.

This way, we also ensure readability and prevent later complications.

To save memory, we made use of get_memory_usage(), which displays all elements in the RAM.

Then, we delete them after use with del and gc.collect().

Because of that, although we define all values at the beginning of each section, we do not delete them at its end for that reason, but for the storage one.

---

If one wishes to use $\delta$ instead of $\delta + 1$, the code is made easy for modification, including the linearization around 0 for any scale and colorbar.

All that is needed is to remove the $+1$ from the grid\_fft\_contrast arrats, uncomment the respective lines and replace the LogNorm with colors.SymLogNorm and "log" with "symlog" and adding the lines below:

In [ ]:
#axs[i00].pcolormesh(X, Y, Z, norm=colors.SymLogNorm(linthresh=0.1, linscale=0.2, vmin=vmin2, vmax=vmax2), cmap='RdBu_r', alpha=1, zorder=1)
#axs[i00].pcolormesh(X, Y, Z, norm=LogNorm(vmin=vmin2, vmax=vmax2), cmap='RdBu_r', alpha=1, zorder=1)

#axs.set_xscale('symlog', linthresh=0.1, linscale=0.2)
#axs.set_xscale('log')


#divider        = make_axes_locatable(axs[j,i])
#cax            = divider.append_axes('right', size='5%', pad=0.05)
#cbar           = plt.colorbar(imm, cax=cax, orientation='vertical')
#tks, tkss = set_ticks(vmin2, vmax2)
#tks.remove(np.float64(-0.01)); tks.remove(np.float64(0.01))
#tkss.remove('$-10^{-2}$'); tkss.remove('$10^{-2}$')
#for i00, label in enumerate(tkss):
#    if   label == r'$-10^{-1}$': tkss[i00] = '\n'+r'$-10^{-1}$'
#    elif label == r'$10^{-1}$':  tkss[i00] = r'$10^{-1}$'+'\n'
#cbar.ax.set_yticks(tks, tkss)
#minor_locator = SymmetricalLogLocator(base=10.0, linthresh=1.0, subs=np.arange(2, 10))
#cbar.ax.yaxis.set_minor_locator(minor_locator)
#cbar.ax.yaxis.set_tick_params(which='minor', length=3, width=0.5, color='k')

---
---
---
---
---
---
---
---
---
---

# 1. Redshift

---
---
---

## 1.1. Redshift vs Age (log vs linear time scale)

---

In [ ]:
sz_indx = 2

Zs_float = ALL_Zs_float[sz_indx]
ages     = ALL_ages[    sz_indx]

---

In [ ]:
H_0 = 70.4   # km/s/Mpc

# 1 Mpc = 3.08568e19 km, 1 Gyr = 3.15576e16 s
H_0_Gyr = H_0 * (3.15576e16 / 3.08568e19)


Omega_r   = 0
Omega_m   = 0.2726
Omega_Lbd = 0.727
Omega_k   = 0

In [ ]:
def fct_H_Z_Gyr(Z):

    return H_0_Gyr * np.sqrt(Omega_r*(1+Z)**4 + Omega_m*(1+Z)**3 + Omega_k*(1+Z)**2 + Omega_Lbd)

In [ ]:
def integrand(Z):
    
    return -1/((1+Z) * fct_H_Z_Gyr(Z))

In [ ]:
def fct_t_Z(Z):

    integral, _ = quad(integrand, np.inf, Z)
    
    return integral

In [ ]:
Zarray = np.arange(Zs_float[0], Zs_float[-1],0.01)

---

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12,4.5), dpi=400, sharey=True)

fig.suptitle(r"Redshift vs Age")

Zs_float[0] = 0.01

loglin = ["(logarithmic", "(linear"]
for i in range(2):

    axs[i].plot(Zarray, [fct_t_Z(_) for _ in Zarray], c='r', label="Calculation", zorder=3)
    imm = axs[i].scatter(Zs_float, ages, label="Illustris labels", zorder=4)
    
    axs[i].set_xlabel('Redshift z '+loglin[i]+" scale)")
    if i == 0:
        xtks, xtkss = set_ticks(Zs_float[0], Zs_float[-1], float_g=True, int_if_possible=True)
        xtkss[0] = "0"
        axs[i].set_xscale('log')
        axs[i].set_xticks(xtks, xtkss)
        axs[i].set_ylabel('Age [Gyr]', labelpad=-5)
    
    if i == 1:
        axs[i].set_xticks([int(_) for _ in Zs_float])
     
    ytks, ytkss = set_ticks(np.min(ages), np.max(ages), log_lin=False, float_g=True, int_if_possible=True)
    axs[i].set_yticks(ytks, ytkss)
    
    xmin, xmax = axs[i].get_xlim()
    x_pos = xmax*0.8
    y_pos = axs[0].get_ylim()[1]*1.01
    
    #if stbl_exp != 0: axs[i].text(x_pos, y_pos, r"$10^{{{0}}}$".format(int(stbl_exp)), ha='center')
    
    divider = make_axes_locatable(axs[i])
    ccc = divider.append_axes('right', size='5%', pad=0.01)
    ccc.set_xticks([], []); ccc.set_yticks([], []); ccc.axis('off')
    
    axs[i].grid()


axs[1].legend()
plt.tight_layout()
plt.savefig(plots_path_0+"1.1___Z_vs_Age.png", bbox_inches='tight')
plt.close()

---
---
---

## 1.2. $\delta_c$ and $\delta_v$

---

In [ ]:
sz_indx = 2

size    = BASELINES[sz_indx]["size"]
Z_fs    = BASELINES[sz_indx]["Z_floatstr"]
nnc     = BASELINES[sz_indx]["nnc"]
R       = BASELINES[sz_indx]["R"]
lvl     = BASELINES[sz_indx]["lvl"]
cl      = BASELINES[sz_indx]["cl"]
uod     = BASELINES[sz_indx]["uod"]; ud, od = uod
uod_str = BASELINES[sz_indx]["uod_str"]
MK      = BASELINES[sz_indx]["MK"]

In [ ]:
BASELINES[2]

In [ ]:
X, Y = np.meshgrid(np.linspace(0, 75, size), np.linspace(0, 75, size)); Y = Y[::-1]

---

In [ ]:
file_path_Z0_MK = "../Modified_Data_"+str(size)+"/Z___00/NNC___"+str(nnc)+"/R___"+str(R)+"/Lvl___"+str(lvl)+"/"+str(cl)+"/D___"+str(uod_str)+"/"+str(MK)+"/"

with open(file_path_Z0_MK+"fg___100.pk", 'rb') as f: fg_100_MK2 = pkl.load(f)

fg_100_MK2_walls = np.ma.masked_where(fg_100_MK2 > -2, fg_100_MK2)

---

In [ ]:
Zs       = ["3.0", "17.0"]
Zs_clean = ["3",   "17"  ]

In [ ]:
file_path_Z3_R  = "../Modified_Data_"+str(size)+"/Z___30/NNC___" +str(nnc)+"/R___"+str(R)+"/"
file_path_Z17_R = "../Modified_Data_"+str(size)+"/Z___170/NNC___"+str(nnc)+"/R___"+str(R)+"/"

with open(file_path_Z3_R +"grid_fft_"+rho_delta+".pk", 'rb') as f: grid_fft_Z3  = pkl.load(f)-1
with open(file_path_Z17_R+"grid_fft_"+rho_delta+".pk", 'rb') as f: grid_fft_Z17 = pkl.load(f)-1

---

In [ ]:
DZ3  = 0.3
DZ17 = 0.073
D_Zs = [DZ3, DZ17]

In [ ]:
grid_fft_Z3  /= DZ3
grid_fft_Z17 /= DZ17

---

In [ ]:
delta_c = -2.81
delta_v =  1.686

In [ ]:
grid_fft_Z3_mask_c  = np.ma.masked_where(grid_fft_Z3  > delta_c, grid_fft_Z3)
grid_fft_Z3_mask_v  = np.ma.masked_where(grid_fft_Z3  < delta_v, grid_fft_Z3)

grid_fft_Z17_mask_c = np.ma.masked_where(grid_fft_Z17 > delta_c, grid_fft_Z17)
grid_fft_Z17_mask_v = np.ma.masked_where(grid_fft_Z17 < delta_v, grid_fft_Z17)

In [ ]:
grid_fft_mask_c = [grid_fft_Z3_mask_c, grid_fft_Z17_mask_c]
grid_fft_mask_v = [grid_fft_Z3_mask_v, grid_fft_Z17_mask_v]
grid_fft        = [grid_fft_Z3,        grid_fft_Z17]

---

In [ ]:
Zs

In [ ]:
cmap_v = LinearSegmentedColormap.from_list('Reds_cut', plt.cm.Reds(np.linspace(0.18,1,256)))
cmap_c = LinearSegmentedColormap.from_list('Blues_r_cut', plt.cm.Blues_r(np.linspace(0.0,0.82,256)))

fig, axs = plt.subplots(1,2, figsize=(12,4.5), dpi=400)

fig.suptitle(  r"Comparing the $D(z)$ prediction on the underdense and overdense regions with the $z=0$ WVF walls." + "\n"
             + r"$yz$-slice [cMpc/h]"
             +  "  |  size="+str(size)
             +  "  |  z="+Z_fs
             + r"  |  $R=$"+str(R)+"cMpc/h"
             + r"  |  Lvl="+latex_float(lvl)
             + r"  |  "+cl
             +  "  |  "+MK
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")


for i in range(2):

    axs[i].set_title("z="+Zs[i]+r"  |  D(z)="+str(D_Zs[i]))
    
    axs[        i].pcolormesh(X, Y, fg_100_MK2_walls[  sc], cmap=ListedColormap(['lime']),                                                                    zorder=2, alpha=0.5)
    
    imm_c = axs[i].pcolormesh(X, Y, grid_fft_mask_c[i][sc], cmap=cmap_c, norm=SymLogNorm(linthresh=1e-4, vmin=np.min(grid_fft[i]), vmax=delta_c),             zorder=1)
    imm_v = axs[i].pcolormesh(X, Y, grid_fft_mask_v[i][sc], cmap=cmap_v, norm=LogNorm(                   vmin=delta_v,             vmax=np.max(grid_fft[i])), zorder=1)

    divider = make_axes_locatable(axs[i])
    cax = divider.append_axes('right', size='5%', pad=0.95)
    cbar = plt.colorbar(imm_v, cax=cax, orientation='vertical')
    tks, tkss = set_ticks(delta_v, np.max(grid_fft[i]), log_lin=True)
    cbar.ax.set_yticks(tks, tkss)
    cbar.set_label(rf'$\delta^{{lim}}_{{{R}\,\mathrm{{cMpc}}/h,{Zs_clean[i]},0}} / D({Zs_clean[i]})$   (in the $\geq \delta_c$ range)', rotation=270, labelpad=15)

    divider = make_axes_locatable(axs[i])

    cax = divider.append_axes('left', size='5%', pad=0.35)
    cbar = plt.colorbar(imm_c, cax=cax, orientation='vertical')
    tks, tkss = set_ticks(np.min(grid_fft[i]), delta_c, log_lin=True)
    cbar.ax.set_yticks(tks, tkss)
    cbar.ax.yaxis.set_ticks_position('left')
    cbar.set_label(rf'$\delta^{{lim}}_{{{R}\,\mathrm{{cMpc}}/h,{Zs_clean[i]},0}} / D({Zs_clean[i]})$   (in the $\leq \delta_v$ range)', rotation=90, labelpad=-55)
    
    axs[i].set_xticks(range_75_5, range_75_5_tkss); axs[i].set_yticks(range_75_5, range_75_5_tkss)

    axs[i].set_aspect('equal')


plt.tight_layout()
plt.savefig(plots_path_0+"1.2___DZ_delta_cv.png", bbox_inches='tight')
plt.close()

---

In [ ]:
del grid_fft, grid_fft_Z3, grid_fft_Z17; gc.collect()

---
---
---
---
---
---
---
---
---
---

# 2. How to plot \& DTFE

---
---
---

## 2.1 Slice plot

---

In [ ]:
# We found this to be a good slice for the 512 frames at Z=00.
# (shows some void origins and contains nice high density regions as well as deep voids).
# sc = slice chosen
# sc = 80

# A value to just show a good and a bad slice is.
sc_trial = [13, sc]

In [ ]:
sz_indx = 2

size = BASELINES[sz_indx]["size"]
Z    = BASELINES[sz_indx]["Z"]
Z_fs = BASELINES[sz_indx]["Z_floatstr"]
nnc  = BASELINES[sz_indx]["nnc"]
R    = BASELINES[sz_indx]["R"]
lvl  = BASELINES[sz_indx]["lvl"]
cl   = BASELINES[sz_indx]["cl"]
uod  = BASELINES[sz_indx]["uod"]; ud, od = uod
MK   = BASELINES[sz_indx]["MK"]

s_r = range(size)

file_path      = "../Modified_Data_"+str(size)+"/"

file_path_Z    = file_path     +"Z___"  +Z                   +"/"
file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"
file_path_Duod = file_path_Lvl +"D___[" +str(ud)+"_"+str(od)+"]/"
file_path_MK   = file_path_Duod+MK                           +"/"

In [ ]:
X, Y = np.meshgrid(np.linspace(0, 75, size), np.linspace(0, 75, size)); Y = Y[::-1]

---

In [ ]:
with open(file_path_R+"grid_fft_"+rho_delta+".pk", 'rb') as f: grid_fft = pkl.load(f)

vmin_fft = np.min(grid_fft); vmax_fft = np.max(grid_fft)

In [ ]:
with open(file_path_MK+"fg___100.pk",  'rb') as f: fg_100 = pkl.load(f)

# Re-order the void indices so they start from 0 and no gaps.
fg_100[fg_100 != -2] = np.searchsorted(np.sort(np.unique(fg_100[fg_100 != -2])), fg_100[fg_100 != -2].ravel())

---

As a rule of thumb, for the x-axis, we use figxize=10 (and general dpi=400) as to keep our figures and their text consistent in the thesis.

In [ ]:
GB = ['Bad features', 'Good features']
alphas = [0, 1, 0.7]

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(12,12.5), dpi=400)

fig.suptitle(  r"Comparing Smoothed sections for features" + "\n"
             + r"$yz$-slice [cMpc/h]"
             +  "  |  size="+str(size)
             +  "  |  z="+Z_fs
             + r"  |  $R=$"+str(R)+"cMpc/h"
             + r"  |  Lvl="+latex_float(lvl)
             + r"  |  "+cl
             +  "  |  "+MK
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")


for i in range(2):
    sc_i = sc_trial[i]

    for j in range(3): axs[j,i].set_xticks(range_75_5, range_75_5_tkss); axs[j,i].set_yticks(range_75_5, range_75_5_tkss)
    
    
    for j in [0,2]:
        grid_fft_sc = grid_fft[sc]
    
        imm = axs[j,i].pcolormesh(X, Y, grid_fft[sc_i], norm=LogNorm(vmin=vmin_fft, vmax=vmax_fft), cmap='RdBu_r', alpha=1, zorder=1)
        axs[j,i].set_aspect('equal')

        if i == 1 and j == 0:
            divider = make_axes_locatable(axs[j,i])
            cax = divider.append_axes('right', size='5%', pad=0.05)
            cbar = plt.colorbar(imm, cax=cax, orientation='vertical')
            tks, tkss = set_ticks(vmin_fft, vmax_fft)
            cbar.ax.set_yticks(tks, tkss)
            ccc = divider.append_axes('right', size='25%', pad=0.1)
            ccc.set_xticks([], []); ccc.set_yticks([], []); ccc.axis('off')

        if i == 1 and j == 0: cbar.set_label(r'$\delta+1$', rotation=270, labelpad=-15)
    

    for j in [1,2]:
        fg_100_sc = fg_100[sc_i]
        fg_100_walls_sc = fg_100_sc == -2
        vmin_fg_100 = 0; vmax_fg_100 = np.max(fg_100)
        
        imm = axs[j,i].imshow(fg_100_sc, cmap='binary', vmin=0, vmax=vmax_fg_100, extent=[0.0, 75, 0.0, 75], alpha=alphas[j], zorder=2)
        axs[j,i].imshow(np.ma.masked_where(~fg_100_walls_sc, fg_100_sc), cmap='winter', extent=[0.0, 75, 0.0, 75], alpha=alphas[j], zorder=2)

        
        if i == 1:
            divider = make_axes_locatable(axs[j,i])
            cax = divider.append_axes('right', size='5%', pad=0.05)
            if j == 1:
                cbar = plt.colorbar(imm, cax=cax, orientation='vertical')
                tks, tkss = set_ticks(vmin_fg_100, vmax_fg_100, log_lin=False, int_if_possible=True)
                cbar.ax.set_yticks(tks, tkss)
            else:
                cax.axis('off')
            ccc = divider.append_axes('right', size='25%', pad=0.1)
            ccc.set_xticks([], []); ccc.set_yticks([], []); ccc.axis('off')
            
        if i == 1: cbar.set_label(r'Void index (reordered)', rotation=270, labelpad=15)
    
    axs[0,i].set_title(GB[i]+r"   $\Delta$x=["+str(round(sc_i/size*75,1))+"-"+str(round((sc_i+1)/size*75,1))+"]cMpc/h")
    axs[1,i].set_title("\n")


plt.tight_layout()
plt.savefig(plots_path_0+"2.1___GB_slice_"+str(R)+".png", bbox_inches='tight')
plt.close()

---

In [ ]:
del X, fg_100_walls_sc; gc.collect()

---
---
---

## 2.2 3D plot

---

In [ ]:
if not os.path.exists(plots_path_0+"2.2___3D_grid_sss"): os.makedirs(plots_path_0+"2.2___3D_grid_sss")

HIGHLY recommended this is not being run at 512...

In [ ]:
sz_indx = 2

size = BASELINES[sz_indx]["size"]
Z    = BASELINES[sz_indx]["Z"]
Z_fs = BASELINES[sz_indx]["Z_floatstr"]
nnc  = BASELINES[sz_indx]["nnc"]
R    = BASELINES[sz_indx]["R"]
lvl  = BASELINES[sz_indx]["lvl"]
cl   = BASELINES[sz_indx]["cl"]
uod  = BASELINES[sz_indx]["uod"]; ud, od = uod
MK   = BASELINES[sz_indx]["MK"]

s_r = range(size)

file_path      = "../Modified_Data_"+str(size)+"/"

file_path_Z    = file_path     +"Z___"  +Z                   +"/"
file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"
file_path_Duod = file_path_Lvl +"D___[" +str(ud)+"_"+str(od)+"]/"
file_path_MK   = file_path_Duod+MK                           +"/"

---

A plot of what density range we use for the 3D plot.

In [ ]:
with open(file_path_R+"grid_fft_"+rho_delta+".pk", 'rb') as f: grid_fft = pkl.load(f)
print(np.min(grid_fft), np.max(grid_fft))

lower_lim = 40
upper_lim = 5380  # 10**1.01

_, _, _ = plt.hist(np.log10(grid_fft).flatten(), bins=200)
plt.axvline(x=np.log10(upper_lim), c="r")
plt.axvline(x=np.log10(lower_lim), c="r")

In [ ]:
@njit(parallel=True)
def lower_upper(grid_fft, lower_lim, upper_lim):

    xyz_x     = []; xyz_y     = []; xyz_z     = []
    
    for i in range(size):
        for j in range(size):
            for k in range(size):
                if lower_lim <= grid_fft[i][j][k] <= upper_lim:
                    xyz_x.append(i); xyz_y.append(j); xyz_z.append(k)

    return xyz_x, xyz_y, xyz_z

In [ ]:
xyz_x, xyz_y, xyz_z = lower_upper(grid_fft, lower_lim, upper_lim)

print(len(xyz_x))

Run this once and restart the notebook maybe... it takes some memory and makes the notebook a bit  unresponsive in the long run.

In [ ]:
plot_3d_interactive(xyz_x, xyz_y, xyz_z)

---

In [ ]:
del xyz_x, xyz_y, xyz_z; gc.collect()

---
---
---

## 2.3. DTFE vs NGP

---

In [ ]:
sz_indx = 2

size = BASELINES[sz_indx]["size"]
Z    = BASELINES[sz_indx]["Z"]
Z_fs = BASELINES[sz_indx]["Z_floatstr"]
nnc  = BASELINES[sz_indx]["nnc"]
R    = BASELINES[sz_indx]["R"]
lvl  = BASELINES[sz_indx]["lvl"]
cl   = BASELINES[sz_indx]["cl"]
uod  = BASELINES[sz_indx]["uod"]; ud, od = uod
MK   = BASELINES[sz_indx]["MK"]

s_r = range(size)

file_path      = "../Modified_Data_"+str(size)+"/"

file_path_Z    = file_path     +"Z___"  +Z                   +"/"
file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"
file_path_Duod = file_path_Lvl +"D___[" +str(ud)+"_"+str(od)+"]/"
file_path_MK   = file_path_Duod+MK                           +"/"

In [ ]:
X, Y = np.meshgrid(np.linspace(0, 75, size), np.linspace(0, 75, size)); Y = Y[::-1]

---

In [ ]:
nncs = [[0,     10**1],
        [10**2, 10**3]]

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12,8.5), dpi=400)

fig.suptitle(r"Comparing the DTFE mehtod with the NGP one" + "\n"
             +  r"$yz$-slice [cMpc/h]"
             +  "  |  size="+str(size)
             +  "  |  z="+Z_fs)

for i in range(2):
    for j in range(2):
        axs[i,j].set_xticks(range_75_5, range_75_5_tkss); axs[i,j].set_yticks(range_75_5, range_75_5_tkss)
        
for i0 in range(2):
    for i1 in range(2):
        nnc = nncs[i0][i1]
    
        file_path_NNC = file_path_Z+"NNC___"+str(nnc)+"/"
        with open(file_path_NNC+"grid.pk", 'rb') as f: grid = pkl.load(f)
        grid *= n_DM / np.sum(grid)
        grid /= np.mean(grid)
        
        vminog = round(np.min(grid),3)
        grid[grid == 0] = 0.1
    
        vmin_grid = np.min(grid); vmax_grid = np.max(grid)
    

        if nnc == 0: axs[i0,i1].set_title("Nearest Grid Point")
        else:        axs[i0,i1].set_title("Number of nearest centroids: "+str(nnc))
        
        imm0 = axs[i0,i1].pcolormesh(X, Y, grid[sc], norm=LogNorm(vmin=vmin_grid, vmax=vmax_grid), cmap='RdBu_r', alpha=1, zorder=1)
        axs[i0,i1].set_aspect('equal')
        
        divider = make_axes_locatable(axs[i0,i1])
        cax = divider.append_axes('right', size='5%', pad=0.05)
        cbar = plt.colorbar(imm0, cax=cax, orientation='vertical')
        tks, tkss = set_ticks(vmin_grid, vmax_grid)
        tkss[0] = str(vminog)
        cbar.ax.set_yticks(tks, tkss)
        cbar.set_label(r"$\delta + 1$", rotation=270, labelpad=-15)


        for xy in range(1,4):
            axs[i0,i1].axvline(x=xy*75/grid_chunks, color='k', alpha=0.3, lw=2, zorder=2)
            axs[i0,i1].axhline(y=xy*75/grid_chunks, color='k', alpha=0.3, lw=2, zorder=2)
    


plt.tight_layout()
plt.savefig(plots_path_0+"2.3___DTFE_vs_NGP.png", bbox_inches='tight')
plt.close()

---

In [ ]:
X, Y = np.meshgrid(np.linspace(0, 18.75, 128), np.linspace(56.25, 75, 128)); Y = Y[::-1]

In [ ]:
range_0_1875      = [0, 18.75*2/4, 18.75*1/4, 18.75*3/4 ,18.75]
range_0_1875_tkss = [0, "", "", "", 18.75]

range_5625_75      = [56.25, 56.25+18.75*2/4, 56.25+18.75*1/4, 56.25+18.75*3/4, 75.]
range_5625_75_tkss = [56.25, "", "", "", 75.]

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12,8.5), dpi=400)

fig.suptitle(r"Comparing the DTFE mehtod with the NGP one (zoomed-in the top left corner)" + "\n"
             +  r"$yz$-slice [cMpc/h]"
             +  "  |  size="+str(size)
             +  "  |  z="+Z_fs)

for i in range(2):
    for j in range(2):
        axs[i,j].set_xticks(range_0_1875, range_0_1875_tkss); axs[i,j].set_yticks(range_5625_75, range_5625_75_tkss)
        
for i0 in range(2):
    for i1 in range(2):
        nnc = nncs[i0][i1]
    
        file_path_NNC = file_path_Z+"NNC___"+str(nnc)+"/"
        with open(file_path_NNC+"grid.pk", 'rb') as f: grid = pkl.load(f)
        grid *= n_DM / np.sum(grid)
        grid /= np.mean(grid)
        vminog = round(np.min(grid),3)
        grid[grid == 0] = 0.1
    
        vmin_grid = np.min(grid); vmax_grid = np.max(grid)
    

        if nnc == 0: axs[i0,i1].set_title("Nearest Grid Point")
        else:        axs[i0,i1].set_title("Number of nearest centroids: "+str(nnc))
        
        imm0 = axs[i0,i1].pcolormesh(X, Y, grid[sc][0:128,0:128], norm=LogNorm(vmin=vmin_grid, vmax=vmax_grid), cmap='RdBu_r', alpha=1, zorder=1)
        axs[i0,i1].set_aspect('equal')
        
        divider = make_axes_locatable(axs[i0,i1])
        cax = divider.append_axes('right', size='5%', pad=0.05)
        cbar = plt.colorbar(imm0, cax=cax, orientation='vertical')
        tks, tkss = set_ticks(vmin_grid, vmax_grid)
        tkss[0] = str(vminog)
        cbar.ax.set_yticks(tks, tkss)
        cbar.set_label(r"$\delta + 1$", rotation=270, labelpad=-15)
    


plt.tight_layout()
plt.savefig(plots_path_0+"2.3___DTFE_vs_NGP_cut.png", bbox_inches='tight')
plt.close()

---
---
---

## 2.4. DTFE Smoothing Filters

---

In [ ]:
sz_indx = 2

size = BASELINES[sz_indx]["size"]
Z    = BASELINES[sz_indx]["Z"]
Z_fs = BASELINES[sz_indx]["Z_floatstr"]
nnc  = BASELINES[sz_indx]["nnc"]
R    = BASELINES[sz_indx]["R"]
lvl  = BASELINES[sz_indx]["lvl"]
cl   = BASELINES[sz_indx]["cl"]
uod  = BASELINES[sz_indx]["uod"]; ud, od = uod
MK   = BASELINES[sz_indx]["MK"]

s_r = range(size)

file_path      = "../Modified_Data_"+str(size)+"/"

file_path_Z    = file_path     +"Z___"  +Z                   +"/"
file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"
file_path_Duod = file_path_Lvl +"D___[" +str(ud)+"_"+str(od)+"]/"
file_path_MK   = file_path_Duod+MK                           +"/"

In [ ]:
filenames = ["../Modified_Data_"+str(size)+"/Z___"+Z+"/NNC___"+str(nnc)+"/R___"+str(R)+"/grid_fft_delta.pk",
             "../Modified_Data_"+str(size)+"/Z___"+Z+"/NNC___"+"0"     +"/R___"+"0.6" +"/grid_fft_delta.pk"]

In [ ]:
titles = [r"Gaussian filter - $R=$"+str(R)+"cMpc/h",
          r"Gaussian filter - $R=0.6$cMpc/h"]

---

In [ ]:
X, Y = np.meshgrid(np.linspace(0, 75, 512), np.linspace(0, 75, 512)); Y = Y[::-1]

In [ ]:
fig, axs = plt.subplots(1,2, figsize=(12,5), dpi=400)

fig.suptitle(r"Comparing the DTFE mehtod with the NGP one after smoothing" + "\n"
             + r"$yz$-slice [cMpc/h]"
             +  "  |  size="+str(size)
             +  "  |  z="+Z_fs)

for i in range(2):

    
    axs[i].set_title(titles[i])
    
    axs[i].set_xticks(range_75_5, range_75_5_tkss); axs[i].set_yticks(range_75_5, range_75_5_tkss)


    with open(filenames[i], 'rb') as f: grid = pkl.load(f)

    vminog = round(np.min(grid),3)
    grid[grid == 0] = 0.1
    vmin_grid = np.min(grid); vmax_grid = np.max(grid)
    imm0 = axs[i].pcolormesh(X, Y, grid[sc], norm=LogNorm(vmin=vmin_grid, vmax=vmax_grid), cmap='RdBu_r', alpha=1, zorder=1)
    
    
    axs[i].set_aspect('equal')

    divider = make_axes_locatable(axs[i])
    cax = divider.append_axes('right', size='5%', pad=0.05)
    cbar = plt.colorbar(imm0, cax=cax, orientation='vertical')
    tks, tkss = set_ticks(vmin_grid, vmax_grid, log_lin=True if i != 2 else False, int_if_possible=False if i != 2 else True)
    tkss[0] = str(vminog)
    cbar.ax.set_yticks(tks, tkss)
    cbar.set_label(r"$\delta+1$", rotation=270, labelpad=-5)
    

plt.tight_layout()
plt.savefig(plots_path_0+"2.4___DTFE_vs_NGP___Smoothing.png", bbox_inches='tight')
plt.close()

---
---
---

## 2.5. DTFE: Parameters

---

In [ ]:
sz_indx = 2

size = BASELINES[sz_indx]["size"]
Z    = BASELINES[sz_indx]["Z"]
Z_fs = BASELINES[sz_indx]["Z_floatstr"]
nnc  = BASELINES[sz_indx]["nnc"]
R    = BASELINES[sz_indx]["R"]

file_path      = "../Modified_Data_"+str(size)+"/"

file_path_Z    = file_path     +"Z___"  +Z                   +"/"
file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"

---

In [ ]:
from scipy.ndimage import median_filter, grey_opening, grey_closing

In [ ]:
X, Y = np.meshgrid(np.linspace(0, 75, 512), np.linspace(0, 75, 512)); Y = Y[::-1]

In [ ]:
titles = [[r"Median filter - size=3",
           r"Grey opening - size=3",
           r"Grey closing - size=3",
           r"Gaussian filter - $R=0.15$cMpc/h"],
          [r"Median filter - size=10",
           r"Grey opening - size=10",
           r"Grey closing - size=10",
           r"Gaussian filter - $R=0.6$cMpc/h"]]

---

In [ ]:
with open(file_path_NNC+"grid.pk", 'rb') as f: grid = pkl.load(f)

---

In [ ]:
density_filtered_2           = median_filter(grid,                 size=3,  mode="wrap")
density_filtered_10          = median_filter(grid,                 size=10, mode="wrap")
density_filtered_210         = [density_filtered_2,     density_filtered_10]

---

In [ ]:
density_filtered_2_2         = grey_opening(density_filtered_2,    size=3,  origin=0, mode="wrap")
density_filtered_2_10        = grey_opening(density_filtered_2,    size=10, origin=0, mode="wrap")
density_filtered_210_210     = [density_filtered_2_2,   density_filtered_2_10]

---

In [ ]:
density_filtered_2_2_2       = grey_closing(density_filtered_2_2,  size=3,  origin=0, mode="wrap")
density_filtered_2_2_10      = grey_closing(density_filtered_2_10, size=10, origin=0, mode="wrap")
density_filtered_210_210_210 = [density_filtered_2_2_2, density_filtered_2_2_10]

---

In [ ]:
density_gaussian_1           = ifftn(fourier_gaussian(fftn(density_filtered_2_2_2), sigma=R  /75*size)).real
density_gaussian_2           = ifftn(fourier_gaussian(fftn(density_filtered_2_2_2), sigma=0.6/75*size)).real
density_gaussian_12          = [density_gaussian_1, density_gaussian_2]

---

In [ ]:
density_filtered_all = [density_filtered_210, density_filtered_210_210, density_filtered_210_210_210, density_gaussian_12]

---

In [ ]:
X, Y = np.meshgrid(np.linspace(45, 60, 103), np.linspace(15, 30, 103)); Y = Y[::-1]

In [ ]:
range_45_60_5 = [45, 50, 55, 60, 65]
range_15_30_5 = [15, 20, 25, 30, 35]

In [ ]:
fig, axs = plt.subplots(4,2, figsize=(12,16.5), dpi=400)

fig.suptitle(r"Comparing the DTFE smoothing parameters. Every second column frame uses the filter parameters" + "\n"
             "(lebeled as size) from the previous (upper) first column one(s)." +"\n"
             + r"$yz$-slice [cMpc/h]"
             +  "  |  size="+str(size)
             +  "  |  z="+Z_fs
             +  "  |  nnc="+str(nnc))

for j in range(2):
    for i in range(4):
        
        axs[i,j].set_title(titles[j][i])
        
        axs[i,j].set_xticks(range_45_60_5, range_45_60_5); axs[i,j].set_yticks(range_15_30_5, range_15_30_5)


        density_filtered  = density_filtered_all[i][j]
        density_filtered *= n_DM / np.sum(density_filtered)
        density_filtered /= np.mean(density_filtered)

        vminog = round(np.min(density_filtered),3)
        density_filtered[density_filtered == 0] = 0.01
        vmin_grid = np.min(density_filtered); vmax_grid = np.max(density_filtered)
        
        imm0 = axs[i,j].pcolormesh(X, Y, density_filtered[sc][512-205:512-102, 307:410], norm=LogNorm(vmin=vmin_grid, vmax=vmax_grid), cmap='RdBu_r', alpha=1, zorder=1)
        axs[i,j].set_aspect('equal')

        divider = make_axes_locatable(axs[i,j])
        cax = divider.append_axes('right', size='5%', pad=0.05)
        cbar = plt.colorbar(imm0, cax=cax, orientation='vertical')
        tks, tkss = set_ticks(vmin_grid, vmax_grid, log_lin=True)
        cbar.ax.set_yticks(tks, tkss)
        tkss[0] = str(vminog)
        cbar.set_label(r"$\delta+1$", rotation=270, labelpad=-5)


plt.tight_layout()
plt.savefig(plots_path_0+"2.5___DTFE_params_3.png", bbox_inches='tight')
plt.close()

In [ ]:
del density_filtered_all, density_filtered_210, density_filtered_210_210, density_filtered_210_210_210, density_gaussian_12, density_filtered_2, density_filtered_10, density_filtered_2_2, density_filtered_2_10, density_filtered_2_2_2, density_filtered_2_2_10, density_filtered, density_gaussian_1, density_gaussian_2; gc.collect()

---
---
---
---
---
---
---
---
---
---

# 3. Smoothing

---
---
---

## 3.1 Grid vs Smoothed frame

---

128
- ~~0.15 - 0.3 cells~~
- ~~0.2  - 0.3 cells~~
- ~~0.3  - 0.5 cells~~
- ~~0.4  - 0.7 cells~~
- 0.6  - 1.0 cells
- 0.8  - 1.4 cells
- 1.0  - 1.7 cells



256
- ~~0.15 - 0.5 cells~~
- ~~0.2  - 0.7 cells~~
- 0.3  - 1.0 cells
- 0.4  - 1.4 cells
- 0.6  - 2.0 cells
- 0.8  - 2.7 cells
- 1.0  - 3.4 cells


512
- 0.15 - 1.0 cells
- 0.2  - 1.4 cells
- 0.3  - 2.0 cells
- 0.4  - 2.7 cells
- 0.6  - 4.1 cells
- 0.8  - 5.5 cells
- 1.0  - 6.8 cells

---

In [ ]:
sz_indx = 2

size = BASELINES[sz_indx]["size"]; d_xyz = 75/size
Z    = BASELINES[sz_indx]["Z"]
Z_fs = BASELINES[sz_indx]["Z_floatstr"]
nnc  = BASELINES[sz_indx]["nnc"]
R    = BASELINES[sz_indx]["R"]

file_path      = "../Modified_Data_"+str(size)+"/"

file_path_Z    = file_path     +"Z___"  +Z                   +"/"
file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"

In [ ]:
X, Y = np.meshgrid(np.linspace(0, 75, size), np.linspace(0, 75, size)); Y = Y[::-1]

---

In [ ]:
with open(file_path_NNC+"grid.pk", 'rb') as f: grid = pkl.load(f)

grid *= m_DM / d_xyz**3
grid /= np.mean(grid)

grid[grid == 0] = 0.01

vmin_grid = np.min(grid); vmax_grid = np.max(grid)
grid_sc = grid[sc]

In [ ]:
with open(file_path_R+"grid_fft_"+rho_delta+".pk", 'rb') as f: grid_fft = pkl.load(f)

vmin_fft = np.min(grid_fft); vmax_fft = np.max(grid_fft)
grid_fft_sc = grid_fft[sc]

In [ ]:
grid_diff = grid - grid_fft

vmin_diff = np.min(grid_diff); vmax_diff = np.max(grid_diff)
grid_diff_sc = grid_diff[sc]

---

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12,8.5), dpi=400)
fig.delaxes(axs[1, 1]); fig.delaxes(axs[1,0])

fig.suptitle(r"Comparing the original grid vs Gaussian smoothened one" + "\n"
             +  r"$yz$-slice [cMpc/h]"
             +  "  |  size="+str(size)
             +  "  |  z="+Z_fs
             + r"  |  $R=$"+str(R)+"cMpc/h")


for i in range(2):
    for j in range(2):
        axs[i,j].set_xticks(range_75_5, range_75_5_tkss); axs[i,j].set_yticks(range_75_5, range_75_5_tkss)



axs[0,0].set_title(r"Original ($\delta_o + 1$)")

imm0 = axs[0,0].pcolormesh(X, Y, grid_sc, norm=LogNorm(vmin=vmin_grid, vmax=vmax_grid), cmap='RdBu_r', alpha=1, zorder=1)
axs[0,0].set_aspect('equal')

divider = make_axes_locatable(axs[0,0])
cax = divider.append_axes('right', size='5%', pad=0.05)
cbar = plt.colorbar(imm0, cax=cax, orientation='vertical')
tks, tkss = set_ticks(vmin_grid, vmax_grid)
cbar.ax.set_yticks(tks, tkss)
minor_locator = SymmetricalLogLocator(base=10.0, linthresh=1.0, subs=np.arange(2, 10))
cbar.ax.yaxis.set_minor_locator(minor_locator)
cbar.ax.yaxis.set_tick_params(which='minor', length=3, width=0.5, color='k')
cbar.set_label(r'$\delta+1$', rotation=270, labelpad=-15)




axs[0,1].set_title(r"Smoothed ($\delta_s + 1$)")

imm1 = axs[0,1].pcolormesh(X, Y, grid_fft_sc, norm=LogNorm(vmin=vmin_fft, vmax=vmax_fft), cmap='RdBu_r', alpha=1, zorder=1)
axs[0,1].set_aspect('equal')

divider = make_axes_locatable(axs[0,1])
cax = divider.append_axes('right', size='5%', pad=0.05)
cbar = plt.colorbar(imm1, cax=cax, orientation='vertical')
tks, tkss = set_ticks(vmin_fft, vmax_fft)
cbar.ax.set_yticks(tks, tkss)
cbar.set_label(r'$\delta+1$', rotation=270, labelpad=-15)




bottom_ax = fig.add_subplot(2, 2, (3, 4)) 
bottom_ax.set_title(r"Original and smoothed difference ($\delta_o - \delta_s$)")

imm2 = bottom_ax.pcolormesh(X, Y, grid_diff_sc, norm=colors.SymLogNorm(linthresh=0.01, linscale=0.2, vmin=vmin_diff, vmax=vmax_diff), cmap='RdBu_r', alpha=1, zorder=1)
bottom_ax.set_aspect('equal')

divider = make_axes_locatable(bottom_ax)
cax = divider.append_axes('right', size='5%', pad=0.05)
cbar = plt.colorbar(imm2, cax=cax, ax=bottom_ax)
tks, tkss = set_ticks(vmin_diff, vmax_diff)
tks.remove(np.float64(-0.01)); tks.remove(np.float64(0.01))
tkss.remove('$-10^{-2}$'); tkss.remove('$10^{-2}$')
cbar.ax.set_yticks(tks, tkss)
cbar.set_label(r'$\delta+1$', rotation=270, labelpad=-15)

bottom_ax.set_xticks(range_75_5, range_75_5_tkss)
bottom_ax.set_yticks(range_75_5, range_75_5_tkss)




plt.tight_layout()
plt.savefig(plots_path_0+"3.1___Original_vs_Smooth.png", bbox_inches='tight')
plt.close()

---

In [ ]:
del grid_diff, X; gc.collect()

---
---
---
---
---
---
---
---
---
---

# 4. Levels and different CDF walls

---
---
---

## 4.1 Leveling

---

In [ ]:
sz_indx = 2

size = BASELINES[sz_indx]["size"]; d_xyz = 75/size
Z    = BASELINES[sz_indx]["Z"]
Z_fs = BASELINES[sz_indx]["Z_floatstr"]
nnc  = BASELINES[sz_indx]["nnc"]
R    = BASELINES[sz_indx]["R"]
lvl  = BASELINES[sz_indx]["lvl"]

file_path      = "../Modified_Data_"+str(size)+"/"

file_path_Z    = file_path     +"Z___"  +Z                   +"/"
file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"

---

In [ ]:
def layer_fct_log(grid_fft, level):
    
    '''
    The complete function for layering.
    '''

    level -= 1
    
    # Make a cube leveled on the log scale for all the steps (even if some are empty).
    grid_fft   = np.log10(grid_fft)
    grid_fft  -= np.min(grid_fft)
    grid_fft  /= np.max(grid_fft)
    grid_fft  *= level
    grid_fft //= 1

    # Remove any empty levels and make the counting continuous (w.r.t. integers).
    value_map = {value: i for i, value in enumerate(np.unique(grid_fft))}
    grid_fft = np.vectorize(value_map.get)(grid_fft)

    # Remove the values that are the max (equal to the max lvl+1).
    # They should be very few but at least one if we didn't remove any empty level, which is highly unlikely.
    # This doesn't matter since size**3 doesn't perfectly divide by level anyway in most cases.
    for i0, j0, k0 in np.argwhere(grid_fft == level): grid_fft[i0][j0][k0] = level-1
    
    return grid_fft

In [ ]:
def layer_fct_cdf(grid_fft, level):
    
    '''
    The complete function for layering.
    '''

    size = grid_fft.shape[0]
    N = size**3
    
    ranks = np.argsort(np.argsort(grid_fft.flatten()))   # elements' ranks
    
    cdf_leveled = np.floor((ranks / N) * level).astype(np.int32)
    cdf_leveled = np.clip(cdf_leveled, 0, level - 1)  # no overflow
    
    grid_lvld = cdf_leveled.reshape((size, size, size))

    return grid_lvld

---

In [ ]:
with open(file_path_R+"grid_fft_"+rho_delta+".pk", 'rb') as f: grid_fft = pkl.load(f)
vmin_fft = np.min(grid_fft); vmax_fft = np.max(grid_fft)

---

In [ ]:
grid_lvld_log = layer_fct_log(grid_fft, lvl)
grid_lvld_cdf = layer_fct_cdf(grid_fft, lvl)

---

In [ ]:
indices_reorder1 = np.argsort(grid_fft.flatten())

In [ ]:
data1 = grid_fft.flatten(     )[indices_reorder1]
data2 = grid_lvld_log.flatten()[indices_reorder1]
data3 = grid_lvld_cdf.flatten()[indices_reorder1]

---

In [ ]:
indices_crop = np.random.choice(len(data1), 10**7, replace=False)

In [ ]:
data1_crop = data1[indices_crop]
data2_crop = data2[indices_crop]
data3_crop = data3[indices_crop]

In [ ]:
indices_reorder1_crop = np.argsort(data1_crop)

In [ ]:
data1_crop = data1_crop[indices_reorder1_crop]
data2_crop = data2_crop[indices_reorder1_crop]
data3_crop = data3_crop[indices_reorder1_crop]

In [ ]:
vmin1 = np.min(data1_crop); vmax1 = np.max(data1_crop)

---

In [ ]:
Z_fs = BASELINES[sz_indx]["Z_floatstr"]

In [ ]:
fig, axs = plt.subplots(figsize=(12,5), dpi=400, sharey=True)

fig.suptitle(r"Levels as a function of (relative) density" + "\n"
             +  "size="+str(size)
             +  "  |  z="+Z_fs
             + r"  |  $R=$"+str(R)+"cMpc/h"
             + r"  |  Lvl="+latex_float(lvl))


axs.plot(data1_crop, data2_crop, lw=2, c="blue", label="log", zorder=3)
axs.plot(data1_crop, data3_crop, lw=2, c="red",  label="cdf", zorder=3)


axs.set_xlabel(r'$\delta + 1$')
axs.set_ylabel(r'Level')

axs.set_xscale('log')

tks, tkss = set_ticks(vmin1, vmax1)
axs.set_xticks(tks, tkss)

axs.legend(loc=4)
axs.grid()
plt.tight_layout()
plt.savefig(plots_path_0+"4.1___Levels_as_density.png", bbox_inches='tight')
plt.close()

---

In [ ]:
del grid_lvld_log, indices_reorder1, data1, data2, data3, data1_crop, data2_crop, indices_reorder1_crop, data3_crop; gc.collect()

---
---
---

## 4.2 CDFs walls 4-frame

---

In [ ]:
sz_indx = 2

size = BASELINES[sz_indx]["size"]; d_xyz = 75/size
Z    = BASELINES[sz_indx]["Z"]
Z_fs = BASELINES[sz_indx]["Z_floatstr"]
nnc  = BASELINES[sz_indx]["nnc"]
R    = BASELINES[sz_indx]["R"]
lvl  = BASELINES[sz_indx]["lvl"]
cl   = BASELINES[sz_indx]["cl"]
MK   = BASELINES[sz_indx]["MK"]

file_path      = "../Modified_Data_"+str(size)+"/"

file_path_Z    = file_path     +"Z___"  +Z                   +"/"
file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"

In [ ]:
X, Y = np.meshgrid(np.linspace(0, 75, size), np.linspace(0, 75, size)); Y = Y[::-1]

In [ ]:
levels_to_check = [10**4, 10**5, 10**6]

---

In [ ]:
with open(file_path_R+"grid_fft_"+rho_delta+".pk", 'rb') as f: grid_fft = pkl.load(f)
vmin_fft = np.min(grid_fft); vmax_fft = np.max(grid_fft)

---

In [ ]:
BASELINES[sz_indx]["uod"]

In [ ]:
ud_od_s = [[[0.1,   0.01], [0.1,   0.02], [0.1,    0.1], [0.1,   0.03]],
           [[0.075, 0.02], [0.1,   0.02], [0.125, 0.02], [0.125, 0.05]]]

In [ ]:
for i0 in range(2):
    
    fig, axs = plt.subplots(2, 2, figsize=(12,8.5), dpi=400)
    axs = axs.flatten()
    
    frame_data = []
    for i1 in range(4):
        
        ud, od = ud_od_s[i0][i1]

        fig.suptitle(  r"Walls & Density: CDF ranges wall comparison" + "\n"
                     + r"$yz$-slice [cMpc/h]"
                     +  "  |  size="+str(size)
                     +  "  |  z="+Z_fs
                     + r"  |  $R=$"+str(R)+"cMpc/h"
                     +  "  |  "+MK)

        if i1 != 1:
            axs[i1].set_title(r"$\Delta_{CDF}=$["+str(ud)+", "+str(od)+"]")
        else:
            axs[i1].set_title("Reference background")

        file_path_Duod = file_path_Lvl + "D___[" + str(ud) + "_" + str(od) + "]/"
        file_path_MK   = file_path_Duod + MK + "/"
        with open(file_path_MK + "fg___100.pk", "rb") as f: grid_100 = pkl.load(f)
        
        mask_ne = (grid_100 != -2)
        uniq = np.sort(np.unique(grid_100[mask_ne]))
        grid_100[mask_ne] = np.searchsorted(uniq, grid_100[mask_ne].ravel())

        grid_100_sc = grid_100[sc]
        neg_mask = (grid_100_sc < 0)
        pos_data = np.ma.masked_where(grid_100_sc < 0, grid_100_sc)

        frame_data.append({"ud":          ud,
                           "od":          od,
                           "grid_100_sc": grid_100_sc,
                           "neg_mask":    neg_mask,
                           "pos_data":    pos_data,
                           "max_val":     np.max(grid_100)})

    # reference is the second plot
    ref_neg_mask = frame_data[1]["neg_mask"]
    
    

    for i1 in range(4):
        ax          = axs[i1]
        ud          = frame_data[i1]["ud"]
        od          = frame_data[i1]["od"]
        grid_100_sc = frame_data[i1]["grid_100_sc"]
        neg_mask    = frame_data[i1]["neg_mask"]
        pos_data    = frame_data[i1]["pos_data"]
        max_val     = frame_data[i1]["max_val"]
        min_val     = 0

        # background
        imm = ax.pcolormesh(X, Y, grid_fft[sc], norm=LogNorm(vmin=vmin_fft, vmax=vmax_fft), cmap="RdBu_r", alpha=1, zorder=1)
        ax.set_aspect("equal")
        if i1 == 1:
            min_val = vmin_fft
            max_val = vmax_fft
            

        else:
            # <0 in both current and reference
            common_mask = neg_mask & ref_neg_mask
            common_img = np.ma.masked_where(~common_mask, common_mask)
            ax.imshow(common_img, vmin=0.1, vmax=1.5, cmap=ListedColormap(["blue"]), extent=[0.0, 75, 0.0, 75], alpha=1, zorder=2)

            # <0 only in current
            unique_cur = neg_mask & ~ref_neg_mask
            unique_cur_img = np.ma.masked_where(~unique_cur, unique_cur)
            ax.imshow(unique_cur_img, vmin=0.1, vmax=1.5, cmap=ListedColormap(["red"]), extent=[0.0, 75, 0.0, 75], alpha=1, zorder=3)

            # <0 only in reference
            unique_ref = ref_neg_mask & ~neg_mask
            unique_ref_img = np.ma.masked_where(~unique_ref, unique_ref)
            ax.imshow(unique_ref_img, vmin=0.1, vmax=1.5, cmap=ListedColormap(["lime"]), extent=[0.0, 75, 0.0, 75], alpha=1, zorder=4)


            # >=0
            imm = ax.imshow(pos_data, vmin=0, vmax=max_val, cmap="binary", extent=[0.0, 75, 0.0, 75], alpha=0.5, zorder=5)
        
        divider = make_axes_locatable(ax)
        cax = divider.append_axes('right', size='5%', pad=0.05)
        
        cbar = plt.colorbar(imm, cax=cax, orientation='vertical')
        tks, tkss = set_ticks(min_val, max_val, log_lin=(False if i1 != 1 else True), int_if_possible=True)
        cbar.ax.set_yticks(tks, tkss)
        ccc = divider.append_axes('right', size='25%', pad=0.1)
        ccc.set_xticks([], []); ccc.set_yticks([], []); ccc.axis('off')
        if i1 != 1:
            cbar.set_label(r'Void index (reordered)', rotation=270, labelpad=15)
        else:
            #cax.axis('off')
            cbar.set_label(r'$\delta+1$', rotation=270, labelpad=-15)

        ax.set_xticks(range_75_5, range_75_5_tkss)
        ax.set_yticks(range_75_5, range_75_5_tkss)



    plt.savefig(plots_path_0+"4.2___CDF_compare_all___"+str(i0)+".png", bbox_inches="tight")
    plt.close()

---

In [ ]:
del mask_ne, X; gc.collect()

---
---
---

## 4.3 Walls creation vs CDF

---

In [ ]:
sz_indx = 2

size = BASELINES[sz_indx]["size"]
Z    = BASELINES[sz_indx]["Z"]
Z_fs = BASELINES[sz_indx]["Z_floatstr"]
nnc  = BASELINES[sz_indx]["nnc"]
R    = BASELINES[sz_indx]["R"]
lvl  = BASELINES[sz_indx]["lvl"]
cl   = BASELINES[sz_indx]["cl"]
uod  = BASELINES[sz_indx]["uod"]; ud, od = uod
MK   = BASELINES[sz_indx]["MK"]

s_r = range(size)

file_path      = "../Modified_Data_"+str(size)+"/"

file_path_Z    = file_path     +"Z___"  +Z                   +"/"
file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"
file_path_Duod = file_path_Lvl +"D___[" +str(ud)+"_"+str(od)+"]/"
file_path_MK   = file_path_Duod+MK                           +"/"

---

In [ ]:
with open(file_path_Lvl+"grid_lvld.pk", 'rb') as f: grid_lvld = pkl.load(f)
with open(file_path_MK+"fg___100.pk", 'rb') as f: fg_100 = pkl.load(f)

In [ ]:
lvl_wals = np.array([grid_lvld[i][j][k] for [i,j,k] in np.argwhere(fg_100 == -2)])
lvl_min = np.min(lvl_wals)
lvl_max = np.max(lvl_wals)
if lvl_max == lvl-1: lvl_max = lvl

In [ ]:
lvl_wals = lvl_wals // int(lvl/1000)
lvl_min /= lvl/100
lvl_max /= lvl/100

In [ ]:
percents, counts = np.unique(lvl_wals, return_counts=True)

In [ ]:
percents = percents / 10

In [ ]:
counts = counts / np.sum(counts) * 100
cdf = np.cumsum(counts)

In [ ]:
fig, axs = plt.subplots(figsize=(12,4), dpi=400)

fig.suptitle(  r"Walls distribution in the grid's CDF" + "\n"
             +       "size="+str(size)
             +  "  |  Z="+Z_fs
             + r"  |  $R=$"+str(R)+"cMpc/h"
             + r"  |  Lvl="+latex_float(lvl)
             + r"  |  "+cl
             +  "  |  "+MK
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")


axs.plot(   percents, cdf, lw=1)
axs.scatter(percents, cdf, s =1, c="r", zorder=3)


grid_CDF_20 = percents[np.argmin(np.abs(cdf - 20))]; grid_CDF_80 = percents[np.argmin(np.abs(cdf - 80))]
axs.hlines(y=20, xmin=-5, xmax=grid_CDF_20, color="green"); axs.vlines(x=grid_CDF_20, ymin=-5, ymax=20, color="green", label="")
axs.hlines(y=80, xmin=-5, xmax=grid_CDF_80, color="green"); axs.vlines(x=grid_CDF_80, ymin=-5, ymax=80, color="green")


xticks  = list(np.linspace(0,100,11)); xticks.append( grid_CDF_20); xticks.append( grid_CDF_80); xticks.append( lvl_min)
xtickss = list(np.linspace(0,100,11)); xtickss.append(grid_CDF_20); xtickss.append(grid_CDF_80); xtickss.append(lvl_min)
for i in range(len(xticks)):
    if xticks[i] not in [grid_CDF_20, grid_CDF_80, lvl_min]:
        if np.abs(xticks[i]-grid_CDF_20) <= 5: xtickss[i] = ""
        if np.abs(xticks[i]-grid_CDF_80) <= 5: xtickss[i] = ""
        if np.abs(xticks[i]-lvl_min)     <= 5: xtickss[i] = ""
axs.set_xticks(xticks, xtickss)
axs.set_xlim(-5,105)
axs.set_ylim(-5,105)

axs.set_xlabel("Grid CDF [%]")
axs.set_ylabel("Walls CDF [%]")

axs.grid()
plt.savefig(plots_path_0+"4.3___Walls_CDF_vs_Grid_CDF.png", bbox_inches="tight")
plt.close()

---
---
---
---
---
---
---
---
---
---

# 5. Origins

---
---
---

## 5.1 Found vs. Lost: levels

---

In [ ]:
sz_indx = 2

Z    = BASELINES[sz_indx]["Z"]
Z_fs = BASELINES[sz_indx]["Z_floatstr"]
nnc  = BASELINES[sz_indx]["nnc"]
cl   = BASELINES[sz_indx]["cl"]

In [ ]:
def myExpFunc(x, a,b):
    return a*np.exp(-b*x)

---

In [ ]:
rb = [["red", "blue", "green"], ["lightcoral", "mediumslateblue", "lime"]]

In [ ]:
fig, axs = plt.subplots(1, 2, dpi=400, figsize=(12,5))

fig.suptitle(  r"MK1 vs MK2: Voids Found and Lost" + "\n"
             +       "size=128 & 256 & 512"
             +  "  |  z="+Z_fs
             + r"  |  "+cl)


ax0_min = 10**7
ax0_max = 0

for sz_indx in range(3):
    
    size      = BASELINES[sz_indx]["size"]
    R         = BASELINES[sz_indx]["R"]
    lvls      = ALL_lvls[ sz_indx][0][0 if sz_indx != 2 else 2][0]

    file_path     = "../Modified_Data_"+str(size)+"/"
    file_path_Z   = file_path    +"Z___"  +Z       +"/"
    file_path_NNC = file_path_Z  +"NNC___"+str(nnc)+"/"
    file_path_R   = file_path_NNC+"R___"  +str(R)  +"/"
    
    
    pair_list = []; isolated_list = []; ratio_list = []
    for i0, lvl in enumerate(lvls):
    
        file_path_Lvl = file_path_R + "Lvl___"+str(lvl)+"/"+cl+"/"

        with open(file_path_Lvl+"dict_origins_isolated.pk", 'rb') as f: dict_origins_isolated = pkl.load(f)
        with open(file_path_Lvl+"dict_origins_pairs.pk",    'rb') as f: dict_origins_pairs    = pkl.load(f)


        len_doi = 0; len_dop = 0
        for i0 in range(len(dict_origins_isolated)): len_doi += len(dict_origins_isolated[i0])
        for i0 in range(len(dict_origins_pairs   )): len_dop += len(dict_origins_pairs[   i0])
        len_dop += len_doi

        if ax0_min > len_doi: ax0_min = len_doi
        if ax0_max < len_dop: ax0_max = len_dop

        isolated_list.append(len_doi)
        pair_list.append(len_dop)
        ratio_list.append(round((1-len_doi/len_dop)*100, 2))
    
    
    axs[0].plot(   lvls, pair_list,     lw=0.6, c=rb[0][sz_indx], zorder=4)
    axs[0].scatter(lvls, pair_list,     s=4,    c=rb[0][sz_indx], label=str(size)+' MK2', zorder=4)
    
    axs[0].plot(   lvls, isolated_list, lw=0.6, c=rb[1][sz_indx], zorder=4)
    axs[0].scatter(lvls, isolated_list, s=4,    c=rb[1][sz_indx], label=str(size)+' MK1', zorder=4)

    ratio_list_print = ratio_list.copy()
    ratio_list = [i if i!=0 else 0.0001 for i in ratio_list]
    popt, pcov = curve_fit(myExpFunc, [np.log10(i) for i in lvls], ratio_list)
    axs[1].plot(   lvls, ratio_list,    lw=0.6, c=rb[0][sz_indx], zorder=4)
    axs[1].scatter(lvls, ratio_list,    s=4,    c=rb[0][sz_indx], label=str(size)+r"  |  "+fr'$\propto e^{{-{round(popt[1],1)} \cdot Lvl}}$', zorder=4)
    
    x_plot_lf = np.linspace(lvls[0], lvls[-1],100000)
    axs[1].plot(x_plot_lf, [myExpFunc(np.log10(i), *popt) for i in x_plot_lf], c=rb[1][sz_indx], lw=0.8, linestyle='--', zorder=4)


for i0 in range(2):
    axs[i0].set_xlabel('Levels')
    axs[i0].set_xscale('log')
    axs[i0].legend()
    axs[i0].grid()  
    
    
axs[0].set_ylim([ax0_min*0.9, ax0_max/0.9])
axs[0].set_yscale("log")
tks, tkss = set_ticks(ax0_min*0.9, ax0_max/0.9, number_of_tks_clean_min=1)
axs[0].set_yticks(tks, tkss)
axs[0].set_title('Voids found')
axs[0].set_ylabel('Counts', labelpad=-15)

axs[1].set_xlim([10**2, 5*10**6])
axs[1].set_ylim([-5, 40])
tks, tkss = set_ticks(-5, 40, log_lin=False, int_if_possible=True)
tkss[0] = " "
axs[1].set_yticks(tks, tkss)
axs[1].set_title('Voids lost from MK2 to MK1')
axs[1].set_ylabel('Percentage [%]')

plt.tight_layout(pad=1.0)
plt.savefig(plots_path_0+"5.1___Compare___voids_found_LVLs.png", bbox_inches='tight')
plt.close()

In [ ]:
del dict_origins_isolated, dict_origins_pairs; gc.collect()

---
---
---

## 5.2 Found vs. Lost: R

---

In [ ]:
sz_indx = 2

Z    = BASELINES[sz_indx]["Z"]
Z_fs = BASELINES[sz_indx]["Z_floatstr"]
nnc  = BASELINES[sz_indx]["nnc"]
cl   = BASELINES[sz_indx]["cl"]

---

In [ ]:
rb = [["red", "blue", "green"], ["lightcoral", "mediumslateblue", "lime"]]

In [ ]:
fig, axs = plt.subplots(1, 2, dpi=400, figsize=(12,5))

fig.suptitle(  r"MK1 vs MK2: Voids Found and Lost" + "\n"
             +       "size=128 & 256 & 512"
             +  "  |  z="+Z_fs
             + r"  |  Lvl=$10^4$ & $10^5$ & $10^5$"
             + r"  |  "+cl)


ax0_min = 10**7
ax0_max = 0

for sz_indx in range(3):
    
    size    = BASELINES[   sz_indx]["size"]
    lvl     = BASELINES[   sz_indx]["lvl"]
    Rs      = ALL_R_cMpchs[sz_indx][0][0 if sz_indx != 2 else 2]

    file_path     = "../Modified_Data_"+str(size)+"/"
    file_path_Z   = file_path   + "Z___"  +Z       +"/"
    file_path_NNC = file_path_Z + "NNC___"+str(nnc)+"/"
    
    
    pair_list = []; isolated_list = []; ratio_list = []
    for i0, R in enumerate(Rs):
    
        file_path_R = file_path_NNC+"R___"  +str(R)+"/"
        file_path_Lvl = file_path_R+"Lvl___"+str(lvl)+"/"+cl+"/"

        with open(file_path_Lvl+"dict_origins_isolated.pk", 'rb') as f: dict_origins_isolated = pkl.load(f)
        with open(file_path_Lvl+"dict_origins_pairs.pk",    'rb') as f: dict_origins_pairs    = pkl.load(f)


        len_doi = 0; len_dop = 0
        for i0 in range(len(dict_origins_isolated)): len_doi += len(dict_origins_isolated[i0])
        for i0 in range(len(dict_origins_pairs   )): len_dop += len(dict_origins_pairs[   i0])
        len_dop += len_doi

        if ax0_min > len_doi: ax0_min = len_doi
        if ax0_max < len_dop: ax0_max = len_dop
        

        isolated_list.append(len_doi)
        pair_list.append(len_dop)
        ratio_list.append(round((1-len_doi/len_dop)*100, 2))
        
    
    axs[0].plot(   Rs, pair_list,     lw=0.6, c=rb[0][sz_indx], zorder=4)
    axs[0].scatter(Rs, pair_list,     s=4,    c=rb[0][sz_indx], label=str(size)+' MK2', zorder=4)
    
    axs[0].plot(   Rs, isolated_list, lw=0.6, c=rb[1][sz_indx])
    axs[0].scatter(Rs, isolated_list, s=4,    c=rb[1][sz_indx], label=str(size)+' MK1', zorder=4)

    ratio_list_print = ratio_list.copy()
    ratio_list = [i if i!=0 else 0.0001 for i in ratio_list]
    axs[1].plot(   Rs, ratio_list,    lw=0.6, c=rb[0][sz_indx], zorder=4)
    axs[1].scatter(Rs, ratio_list,    s=4,    c=rb[0][sz_indx], label=str(size), zorder=4)
    

for i0 in range(2):
    Rss = Rs.copy()
    Rss[0] = str(Rss[0])+"  "
    axs[i0].set_xticks(Rs, Rss)
    axs[i0].set_xlabel(r'Gaussian smoothing parameter $R$')
    axs[i0].legend()
    axs[i0].grid()  
    

axs[0].set_ylim([ax0_min*0.9, ax0_max/0.9])#[-1000, 30000])
axs[0].set_yscale("log")
tks, tkss = set_ticks(ax0_min*0.9, ax0_max/0.9, number_of_tks_clean_min=1)
axs[0].set_yticks(tks, tkss)
axs[0].set_title('Voids found')
axs[0].set_ylabel('Counts', labelpad=-15)

axs[1].set_ylim([-0.6, 20])
tks, tkss = set_ticks(0, 20, log_lin=False, int_if_possible=True)
axs[1].set_yticks(tks, tkss)
axs[1].set_title('Voids lost from MK2 to MK1')
axs[1].set_ylabel('Percentage [%]')

plt.tight_layout(pad=1.0)
plt.savefig(plots_path_0+"5.2___Compare___voids_found_Rs.png", bbox_inches='tight')
plt.close()

---

In [ ]:
del dict_origins_pairs, dict_origins_isolated; gc.collect()

---
---
---

## 5.3 Origins density distribution

---

In [ ]:
sz_indx = 2

size = BASELINES[sz_indx]["size"]; d_xyz = 75/size
Z    = BASELINES[sz_indx]["Z"]
Z_fs = BASELINES[sz_indx]["Z_floatstr"]
nnc  = BASELINES[sz_indx]["nnc"]
R    = BASELINES[sz_indx]["R"]
cl   = BASELINES[sz_indx]["cl"]

file_path      = "../Modified_Data_"+str(size)+"/"

file_path_Z    = file_path     +"Z___"  +Z                   +"/"
file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"

In [ ]:
with open(file_path_R+"grid_fft_"+rho_delta+".pk", 'rb') as f: grid_fft = pkl.load(f)

---

In [ ]:
vmin_list = []; vmax_list = []
for i0, lvl in enumerate(ALL_lvls[sz_indx][0][2][0][1:]):    # [3000, 10000, 30000, 100000, 300000, 1000000]
    
    file_path_Z   = file_path    +"Z___"  +Z       +"/"
    file_path_NNC = file_path_Z  +"NNC___"+str(nnc)+"/"
    file_path_R   = file_path_NNC+"R___"  +str(R)  +"/"
    file_path_Lvl = file_path_R  +"Lvl___"+str(lvl)+"/"+cl+"/"

    with open(file_path_Lvl+"dict_origins_isolated.pk", 'rb') as f: dict_origins_isolated = pkl.load(f)
    with open(file_path_Lvl+"dict_origins_pairs.pk",    'rb') as f: dict_origins_pairs    = pkl.load(f)

    origins_isolated_vals = []; origins_pairs_vals = []
    for lvl_i in range(lvl):
        for i00 in range(len(dict_origins_isolated[lvl_i])):
            ii0,jj0,kk0 = dict_origins_isolated[lvl_i][i00]
            origins_isolated_vals.append(grid_fft[ii0][jj0][kk0])
        for i00 in range(len(dict_origins_pairs[   lvl_i])):
            ii0,jj0,kk0 = dict_origins_pairs[lvl_i][i00][0]
            origins_pairs_vals.append(   grid_fft[ii0][jj0][kk0])
    
    vmin_list.append(np.min(origins_isolated_vals)); vmax_list.append(np.max(origins_pairs_vals))

vmin = np.min(vmin_list); vmax = np.max(vmax_list)

In [ ]:
bins_log = np.logspace(np.log10(vmin), np.log10(vmax), 50)

---

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(12,12), dpi=400)

fig.suptitle(  r"Origins distribution as a function of density: isolated vs pairs" + "\n"
             +       "size="+str(size)
             +  "  |  z="+Z_fs
             + r"  |  $R=$"+str(R)+"cMpc/h"
             + r"  |  "+cl)


for i0, lvl in enumerate(ALL_lvls[sz_indx][0][2][0][1:]):    # [3000, 10000, 30000, 100000, 300000, 1000000]
    
    file_path_Z   = file_path    +"Z___"  +Z       +"/"
    file_path_NNC = file_path_Z  +"NNC___"+str(nnc)+"/"
    file_path_R   = file_path_NNC+"R___"  +str(R)  +"/"
    file_path_Lvl = file_path_R  +"Lvl___"+str(lvl)+"/"+cl+"/"

    with open(file_path_Lvl+"dict_origins_isolated.pk", 'rb') as f: dict_origins_isolated = pkl.load(f)
    with open(file_path_Lvl+"dict_origins_pairs.pk",    'rb') as f: dict_origins_pairs    = pkl.load(f)

    origins_isolated_vals = []; origins_pairs_vals = []
    for lvl_i in range(lvl):
        for i00 in range(len(dict_origins_isolated[lvl_i])):
            ii0,jj0,kk0 = dict_origins_isolated[lvl_i][i00]
            origins_isolated_vals.append(grid_fft[ii0][jj0][kk0])
        for i00 in range(len(dict_origins_pairs[lvl_i])):
            ii0,jj0,kk0 = dict_origins_pairs[lvl_i][i00][0]
            origins_pairs_vals.append(grid_fft[ii0][jj0][kk0])
    
    len_origins = len(origins_isolated_vals) + len(origins_pairs_vals)


    
    # Normalize
#    fig1, axs1 = plt.subplots(dpi=50)
#    y2, x2, _ = axs1.hist([origins_isolated_vals, origins_pairs_vals], stacked=True, bins=bins_log)
#    plt.close()

#    if i0 == 1: axs[i0//2,i0%2].hist([origins_isolated_vals, origins_pairs_vals], stacked=True, bins=bins_log,
#                                       weights=[[1/1/np.sum(y2[1])] * len(origins_isolated_vals), [1/1/np.sum(y2[1])] * len(origins_pairs_vals)],
#                                       alpha=1, edgecolor='black', linewidth=0.2, label=['isolated', 'pairs'], zorder=3)
#    else:       axs[i0//2,i0%2].hist([origins_isolated_vals, origins_pairs_vals], stacked=True, bins=bins_log,
#                                       weights=[[1/1/np.sum(y2[1])] * len(origins_isolated_vals), [1/1/np.sum(y2[1])] * len(origins_pairs_vals)],
#                                       alpha=1, edgecolor='black', linewidth=0.2, zorder=3)

    if i0 == 1: axs[i0//2,i0%2].hist([origins_isolated_vals, origins_pairs_vals], stacked=True, bins=bins_log,
                                       alpha=1, edgecolor='black', linewidth=0.2, label=['isolated', 'pairs'], zorder=3)
    else:       axs[i0//2,i0%2].hist([origins_isolated_vals, origins_pairs_vals], stacked=True, bins=bins_log,
                                       alpha=1, edgecolor='black', linewidth=0.2, zorder=3)
    
    axs[i0//2,i0%2].set_xscale('log')

    axs[i0//2,i0%2].text(0.975, 0.6, 
                     "Percentage of pairs\nfrom total: "+str(round(100*len(origins_pairs_vals)/len_origins,2))+"%", 
                     transform=axs[i0//2,i0%2].transAxes, 
                     fontsize=10, 
                     horizontalalignment='right', 
                     verticalalignment='center', 
                     bbox=dict(facecolor='white', edgecolor='grey'))

    axs[i0//2,i0%2].set_title("Lvl="+latex_float(lvl, int_base_if_possible=True))
    axs[i0//2,i0%2].grid()



tks, tkss = set_ticks(vmin, vmax)
for i0 in range(len(ALL_lvls[sz_indx][0][2][0][1:])):
    axs[i0//2,i0%2].set_xticks(tks, tkss)
    axs[i0//2,i0%2].set_ylabel("Origin counts")
    axs[i0//2,i0%2].set_xlabel(r"$\delta+1$", labelpad=0)

axs[0,1].legend(loc=1)
plt.tight_layout()
plt.savefig(plots_path_0+"5.3___Origins_density_distribution___Lvl.png", bbox_inches='tight')
plt.close()

---

In [ ]:
del dict_origins_isolated, dict_origins_pairs, origins_isolated_vals; gc.collect()

---
---
---

## 5.4 Void Origins: start vs end

---

In [ ]:
sz_indx = 2

size    = BASELINES[sz_indx]["size"]; d_xyz = 75/size
Z       = BASELINES[sz_indx]["Z"]
Z_fs    = BASELINES[sz_indx]["Z_floatstr"]
nnc     = BASELINES[sz_indx]["nnc"]
R       = BASELINES[sz_indx]["R"]
lvl     = BASELINES[sz_indx]["lvl"]
cl      = BASELINES[sz_indx]["cl"]
uod     = BASELINES[sz_indx]["uod"]; ud, od = uod
MK      = BASELINES[sz_indx]["MK"]

file_path      = "../Modified_Data_"+str(size)+"/"

file_path_Z    = file_path     +"Z___"  +Z                   +"/"
file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"
file_path_Duod = file_path_Lvl +"D___[" +str(ud)+"_"+str(od)+"]/"
file_path_MK   = file_path_Duod+MK                           +"/"

---

In [ ]:
with open(file_path_Lvl+"dict_origins_isolated.pk", 'rb') as f: dict_origins_isolated = pkl.load(f)
with open(file_path_Lvl+"dict_origins_pairs.pk",    'rb') as f: dict_origins_pairs    = pkl.load(f)

In [ ]:
with open(file_path_Lvl+"ud_val___"+str(ud)+".pk", 'rb') as f: ud_val = pkl.load(f)
with open(file_path_Lvl+"od_val___"+str(od)+".pk", 'rb') as f: od_val = pkl.load(f)

In [ ]:
with open(file_path_Lvl+"ud_lvl___"+str(ud)+".pk", 'rb') as f: ud_lvl = pkl.load(f)
with open(file_path_Lvl+"od_lvl___"+str(od)+".pk", 'rb') as f: od_lvl = pkl.load(f)

---

In [ ]:
with open(file_path_R  +"grid_fft_"+rho_delta+".pk", 'rb') as f: grid_fft = pkl.load(f)
with open(file_path_Lvl+"grid_lvld.pk",              'rb') as f: grid_lvld = pkl.load(f)

In [ ]:
with open(file_path_Lvl+"fg___0_"+MK+".pk", 'rb') as f: fg_0_MK2   = pkl.load(f)
with open(file_path_MK +"fg___100.pk",      'rb') as f: fg_100_MK2 = pkl.load(f)

In [ ]:
unique_fg_100_MK2 = np.unique(fg_100_MK2)

---

In [ ]:
od_lvl

In [ ]:
len_odi_origins = 0; len_odp_origins = 0
for i0 in range(od_lvl):
    len_odi_origins += len(dict_origins_isolated[i0])
    len_odp_origins += len(dict_origins_pairs[   i0])

len_odi_origins, len_odp_origins, len_odi_origins+len_odp_origins, len_odp_origins/(len_odi_origins+len_odp_origins)*100

---

In [ ]:
start_oi_fft = []; end_oi_fft = []
start_op_fft = []; end_op_fft = []
start_oi_lvl = []; end_oi_lvl = []
start_op_lvl = []; end_op_lvl = []

for lvl_i in range(lvl):
    len_doi = len(dict_origins_isolated[lvl_i])
    len_dop = len(dict_origins_pairs[   lvl_i])
    
    for i0 in range(len_doi):
        i00, j00, k00 = dict_origins_isolated[lvl_i][i0]
        val_fft = grid_fft[ i00][j00][k00]; start_oi_fft.append(val_fft)
        val_lvl = grid_lvld[i00][j00][k00]; start_oi_lvl.append(val_lvl)
        
        # If this void index still exists in the final filled grid....
        if fg_0_MK2[i00][j00][k00] == fg_100_MK2[i00][j00][k00]:
            end_oi_fft.append(val_fft)
            end_oi_lvl.append(val_lvl)
    
    for i0 in range(len_dop):
        i00, j00, k00 = dict_origins_pairs[lvl_i][i0][0]
        val_fft = grid_fft[ i00][j00][k00]; start_op_fft.append(val_fft)
        val_lvl = grid_lvld[i00][j00][k00]; start_op_lvl.append(val_lvl)
        
        # If this void index still exists in the final filled grid....
        if fg_0_MK2[i00][j00][k00] == fg_100_MK2[i00][j00][k00]:
            end_op_fft.append(val_fft)
            end_op_lvl.append(val_lvl)

---

In [ ]:
vmin = np.min(grid_fft)
vmax = np.max(grid_fft)

In [ ]:
N_bins = 300

log_vmin = np.log10(vmin)
log_vmax = np.log10(vmax)
log_od   = np.log10(od_val)


dlog_old     = (log_vmax - log_vmin) / N_bins
n_od_float   = (log_od - log_vmin) / dlog_old
n_od         = int(np.ceil(n_od_float))
dlog_new     = (log_od - log_vmin) / n_od
log_vmax_new = log_vmin + N_bins * dlog_new
vmax_new     = 10**log_vmax_new

bins_log = np.logspace(log_vmin, log_vmax_new, N_bins + 1)
bins_log[n_od] = od_val

In [ ]:
perc_start = str(round(len(start_op_fft)/(len(start_op_fft)+len(start_oi_fft))*100,2))
perc_end   = str(round(len(end_op_fft  )/(len(end_op_fft  )+len(end_oi_fft  ))*100,2))

---

In [ ]:
# We found this to be a good limit given our RAM.
max_lvls_per_list = round(lvl / 100 * (512/size)**3)
no_lists = lvl // max_lvls_per_list + (1 if (lvl % max_lvls_per_list)!=0 else 0)

In [ ]:
cells_isolated = []; cells_pairs = []
for i in range(no_lists):
    
    with open(file_path_Lvl+"dict_cells_isolated/"+str(i)+".pk", 'rb') as f: dict_cells_isolated = pkl.load(f)
    with open(file_path_Lvl+"dict_cells_pairs/"+str(i)   +".pk", 'rb') as f: dict_cells_pairs    = pkl.load(f)

    start_lvl = max_lvls_per_list*i
    end_lvl   = np.min([max_lvls_per_list*(i+1), lvl])

    for current_lvl in range(start_lvl, end_lvl):

        isolated_i = np.array(dict_cells_isolated[current_lvl])
        if len(isolated_i) != 0:
            for [i,j,k] in isolated_i: cells_isolated.append(grid_fft[i][j][k])
                


        pairs = dict_cells_pairs[current_lvl]
        if len(pairs) != 0:
            for pairs_i in pairs:
                pairs_i = np.array(pairs_i)
                for [i,j,k] in pairs_i: cells_pairs.append(grid_fft[i][j][k])

In [ ]:
perc_grid = round(len(cells_pairs) / (len(cells_isolated)+len(cells_pairs)) * 100, 2)

---

In [ ]:
x_cut = 10**((np.log10(vmin) + np.log10(vmax)) / 2)

fig = plt.figure(figsize=(12,7.2), dpi=400)
gs  = fig.add_gridspec(2, 2, height_ratios=[1,1], width_ratios=[1,1])

axs = [fig.add_subplot(gs[0, :]),
       fig.add_subplot(gs[1, 0]),
       fig.add_subplot(gs[1, 1])]

fig.suptitle(  r"Grid and origins distributions as a function of density: remaining after Finder completion" + "\n"
             +       "size="+str(size)
             +  "  |  z="+Z_fs
             + r"  |  $R=$"+str(R)+"cMpc/h"
             + r"  |  Lvl="+latex_float(lvl)
             + r"  |  "+cl
             +  "  |  "+MK
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")

axs[0].hist([cells_isolated, cells_pairs], stacked=True, bins=bins_log, alpha=1, color=['#1f77b4', '#ff7f0e'], edgecolor='black', linewidth=0.2, 
            label=['Grid isolated cells', 'Grid pairs: '+str(perc_grid)+"%"], zorder=3)

axs[1].hist([start_oi_fft, start_op_fft], stacked=True, bins=bins_log, alpha=1, color=['#1f77b4', '#ff7f0e'], edgecolor='black', linewidth=0.2, 
            label=['Start isolated: '  +str(len(start_oi_fft)), 'Start pairs:      '+str(len(start_op_fft))+" | "+perc_start+"%"], zorder=3)

axs[2].hist([end_oi_fft,   end_op_fft  ], stacked=True, bins=bins_log, alpha=1, color=['#1f57b4', '#ff5f0e'], edgecolor='black', linewidth=0.2, 
            label=['End isolated: '+str(len(end_oi_fft  )), 'End pairs:      '+str(len(end_op_fft  ))+" | "+perc_end  +"%"], zorder=3)

tks, tkss = set_ticks(vmin, vmax)

for i0 in range(3):
    axs[i0].axvline(ud_val, c="lime", lw=1.5, label=r"$\Delta_{CDF}=$["+str(ud)+","+str(od)+"]", zorder=4)
    axs[i0].axvline(od_val, c="lime", lw=1.5, zorder=4)

    axs[i0].set_xscale('log')
    axs[i0].set_xticks(tks, tkss)
    axs[i0].legend()
    axs[i0].grid()

axs[0].set_xlim(vmin, vmax)
axs[1].set_xlim(vmin, x_cut)
axs[2].set_xlim(vmin, x_cut)



axs[1].set_xlabel(r"$\delta+1$", labelpad=0)
axs[2].set_xlabel(r"$\delta+1$", labelpad=0)

axs[0].set_ylabel("Grid cell counts")
axs[1].set_ylabel("Origin counts")
axs[2].set_ylabel("Origin counts")

plt.tight_layout()
plt.savefig(plots_path_0+"5.4___Origins_density_distribution___Remaining.png", bbox_inches='tight')
plt.close()

---
---

In [ ]:
vmin_fft = 10**9; vmax_fft = -1
vmin_lvl = 0;     vmax_lvl = lvl
for Z in ALL_Zs[sz_indx][:1]:

    file_path_Z    = file_path     +"Z___"  +Z                   +"/"
    file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
    file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
    file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"
    file_path_Duod = file_path_Lvl +"D___[" +str(ud)+"_"+str(od)+"]/"
    file_path_MK   = file_path_Duod+MK                           +"/"
    
    with open(file_path_Lvl+"dict_origins_isolated.pk", 'rb') as f: dict_origins_isolated = pkl.load(f)
    with open(file_path_Lvl+"dict_origins_pairs.pk",    'rb') as f: dict_origins_pairs    = pkl.load(f)
    
    with open(file_path_R  +"grid_fft_"+rho_delta+".pk", 'rb') as f: grid_fft  = pkl.load(f)

    min_fft = np.min(grid_fft); max_fft = np.max(grid_fft)
    if min_fft < vmin_fft: vmin_fft = min_fft
    if max_fft > vmin_fft: vmax_fft = max_fft

In [ ]:
bins_log = np.logspace(np.log10(vmin_fft), np.log10(vmax_fft), N_bins+1)
bins_log_fft = (bins_log[1:]+bins_log[:-1])/2

In [ ]:
profiles_grid_fft = []; profiles_origins_fft = []
count_oi = []; count_op = []
ud_vals = []; ud_grid_intersections = []; ud_origins_intersections = []
od_vals = []; od_grid_intersections = []; od_origins_intersections = []

for Z in ALL_Zs[sz_indx]:
    print(Z)

    file_path_Z    = file_path     +"Z___"  +Z                   +"/"
    file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
    file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
    file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"
    file_path_Duod = file_path_Lvl +"D___[" +str(ud)+"_"+str(od)+"]/"
    file_path_MK   = file_path_Duod+MK                           +"/"
    
    with open(file_path_Lvl+"dict_origins_isolated.pk", 'rb') as f: dict_origins_isolated = pkl.load(f)
    with open(file_path_Lvl+"dict_origins_pairs.pk",    'rb') as f: dict_origins_pairs    = pkl.load(f)

    with open(file_path_Lvl+"ud_val___"+str(ud)+".pk", 'rb') as f: ud_val = pkl.load(f)
    with open(file_path_Lvl+"od_val___"+str(od)+".pk", 'rb') as f: od_val = pkl.load(f)
    ud_vals.append(ud_val); od_vals.append(od_val)
    
    with open(file_path_R  +"grid_fft_"+rho_delta+".pk", 'rb') as f: grid_fft  = pkl.load(f)

    
    cells_origins_fft = []
    count_oi.append(0); count_op.append(0)
    for lvl_i in range(lvl):
        len_doi = len(dict_origins_isolated[lvl_i])
        len_dop = len(dict_origins_pairs[   lvl_i])
        
        for i0 in range(len_doi):
            i00, j00, k00 = dict_origins_isolated[lvl_i][i0]
            cells_origins_fft.append(grid_fft[ i00][j00][k00])
            count_oi[-1] += 1
        
        for i0 in range(len_dop):
            i00, j00, k00 = dict_origins_pairs[lvl_i][i0][0]
            cells_origins_fft.append(grid_fft[ i00][j00][k00])
            count_op[-1] += 1

    

    arabud = np.argmin(np.abs(bins_log_fft-ud_val))
    arabod = np.argmin(np.abs(bins_log_fft-od_val))

    plt.figure()
    hist_vals = plt.hist(cells_origins_fft, bins=bins_log)
    plt.xscale('log')
    plt.close()
    profiles_origins_fft.append(    hist_vals[0])
    ud_origins_intersections.append(hist_vals[0][arabud])
    od_origins_intersections.append(hist_vals[0][arabod])
    
    cells_grid_fft = grid_fft.flatten()
    plt.figure()
    hist_vals = plt.hist(cells_grid_fft, bins=bins_log)
    plt.xscale('log')
    plt.close()
    profiles_grid_fft.append(    hist_vals[0])
    ud_grid_intersections.append(hist_vals[0][arabud])
    od_grid_intersections.append(hist_vals[0][arabod])

In [ ]:
with open(analysis_path+"bins_log_fft.pk",             'wb') as f: pkl.dump(bins_log_fft,             f)
with open(analysis_path+"profiles_grid_fft.pk",        'wb') as f: pkl.dump(profiles_grid_fft,        f)
with open(analysis_path+"ud_vals.pk",                  'wb') as f: pkl.dump(ud_vals,                  f)
with open(analysis_path+"od_vals.pk",                  'wb') as f: pkl.dump(od_vals,                  f)
with open(analysis_path+"ud_grid_intersections.pk",    'wb') as f: pkl.dump(ud_grid_intersections,    f)
with open(analysis_path+"od_grid_intersections.pk",    'wb') as f: pkl.dump(od_grid_intersections,    f)
with open(analysis_path+"ud_origins_intersections.pk", 'wb') as f: pkl.dump(ud_origins_intersections, f)
with open(analysis_path+"od_origins_intersections.pk", 'wb') as f: pkl.dump(od_origins_intersections, f)

with open(analysis_path+"profiles_origins_fft.pk",     'wb') as f: pkl.dump(profiles_origins_fft,     f)
with open(analysis_path+"profiles_grid_fft.pk",        'wb') as f: pkl.dump(profiles_grid_fft,        f)

---

In [ ]:
del cells_isolated, Out, cells_grid_fft, cells_pairs, dict_cells_pairs, dict_cells_isolated, dict_origins_isolated, dict_origins_pairs, cells_origins_fft; gc.collect()

---
---
---
---
---
---
---
---
---
---

# 6. CDFs

From now on, we just call grid_fft_contrast simply grid_fft.

---
---
---

## 6.1 CDF Origins vs CDF Gaussian

---

In [ ]:
sz_indx = 2

size    = BASELINES[sz_indx]["size"]; d_xyz = 75/size
nnc     = BASELINES[sz_indx]["nnc"]
R       = BASELINES[sz_indx]["R"]
lvl     = BASELINES[sz_indx]["lvl"]
cl      = BASELINES[sz_indx]["cl"]
uod     = BASELINES[sz_indx]["uod"]; ud, od = uod

file_path      = "../Modified_Data_"+str(size)+"/"

---

In [ ]:
# the list of density values of each origin (at each level)
doip_fft_vals_list_all = []

for zP_index, Z in enumerate(ALL_Zs[sz_indx]):

    file_path_Z   = file_path    +"Z___"  +Z       +"/"
    file_path_NNC = file_path_Z  +"NNC___"+str(nnc)+"/"
    file_path_R   = file_path_NNC+"R___"  +str(R)  +"/"
    file_path_Lvl = file_path_R  +"Lvl___"+str(lvl)+"/"+cl+"/"

    with open(file_path_R+"grid_fft_"+rho_delta+".pk", 'rb') as f: grid_fft = pkl.load(f)

    with open(file_path_Lvl+"dict_origins_isolated.pk", 'rb') as f: dict_origins_isolated = pkl.load(f)
    with open(file_path_Lvl+"dict_origins_pairs.pk",    'rb') as f: dict_origins_pairs    = pkl.load(f)
    
    doip_fft_vals_list = []
    for lvl_index in range(len(dict_origins_isolated)):
        doi_lvl = dict_origins_isolated[lvl_index]
        for doi_i in doi_lvl:
            ii0, jj0, kk0 = doi_i
            doip_fft_vals_list.append(grid_fft[ii0][jj0][kk0])
        dop_lvl = dict_origins_pairs[lvl_index]
        for dop_i in dop_lvl:
            ii0, jj0, kk0 = dop_i[0]
            doip_fft_vals_list.append(grid_fft[ii0][jj0][kk0]) # ok we don't take the avg but cmmon it's fine...
    doip_fft_vals_list_all.append(np.array(doip_fft_vals_list))

with open(analysis_path+"doip_fft_vals_list_all.pk", 'wb') as f: pkl.dump(doip_fft_vals_list_all,  f)

---

In [ ]:
with open(analysis_path+"doip_fft_vals_list_all.pk", 'rb') as f: doip_fft_vals_list_all  = pkl.load(f)

In [ ]:
cdf_vals_list_fft = []; cdf_vals_list_doip = []
vmm_log_list = []

for Z_index, Z in enumerate(ALL_Zs[sz_indx]):
    print(Z)

    file_path_Z   = file_path    +"Z___"  +Z       +"/"
    file_path_NNC = file_path_Z  +"NNC___"+str(nnc)+"/"
    file_path_R   = file_path_NNC+"R___"  +str(R)  +"/"
    file_path_Lvl = file_path_R  +"Lvl___"+str(lvl)+"/"+cl+"/"

    with open(file_path_R+"grid_fft_"+rho_delta+".pk", 'rb') as f: grid_fft = pkl.load(f)
    
    doip_fft_vals_list = doip_fft_vals_list_all[Z_index]


    for i22 in range(2):
        if i22 == 0:
            data_log = np.log10(grid_fft.flatten())
            vmin_log = np.min(data_log); vmax_log = np.max(data_log)
            vmm_log_list.append([vmin_log, vmax_log])
            x_vals_log = np.linspace(vmin_log, vmax_log, 10**4)
        else:
            data_log = np.log10(doip_fft_vals_list)
        
        density_vals, x_density = fastKDE_pdf(data_log, num_points=2**10+1)
        #x_density = np.linspace(np.min(data_log), np.max(data_log), len(density_vals))

        # we interpolate both CDFs on the scale of the grid's CDF
        density_vals_interp  = np.interp(x_vals_log, x_density, density_vals)
        density_vals_interp /= np.sum(density_vals_interp)
        
        cdf_vals  = cumulative_trapezoid(density_vals_interp, x_vals_log, initial=0)
        cdf_vals /= cdf_vals[-1]
        
        if i22 == 0: cdf_vals_list_fft.append(cdf_vals)
        else:        cdf_vals_list_doip.append(cdf_vals)

with open(analysis_path+"cdf_vals_list_fft.pk",  'wb') as f: pkl.dump(cdf_vals_list_fft,  f)
with open(analysis_path+"cdf_vals_list_doip.pk", 'wb') as f: pkl.dump(cdf_vals_list_doip, f)
with open(analysis_path+"vmm_log_list.pk",       'wb') as f: pkl.dump(vmm_log_list,       f)

In [ ]:
ud_val_list = []; od_val_list = []
for Z_index, Z in enumerate(ALL_Zs[sz_indx]):

    file_path_Z   = file_path    +"Z___"  +Z       +"/"
    file_path_NNC = file_path_Z  +"NNC___"+str(nnc)+"/"
    file_path_R   = file_path_NNC+"R___"  +str(R)  +"/"
    file_path_Lvl = file_path_R  +"Lvl___"+str(lvl)+"/"+cl+"/"
    
    with open(file_path_Lvl+"ud_val___"+str(ud)+".pk", 'rb') as f: ud_val_saved = pkl.load(f)
    with open(file_path_Lvl+"od_val___"+str(od)+".pk", 'rb') as f: od_val_saved = pkl.load(f)

    ud_val_list.append(ud_val_saved)
    od_val_list.append(od_val_saved)

uod_val_list = np.unique(ud_val_list + od_val_list)

with open(analysis_path+"ud_val_list.pk",        'wb') as f: pkl.dump(ud_val_list,        f)
with open(analysis_path+"od_val_list.pk",        'wb') as f: pkl.dump(od_val_list,        f)
with open(analysis_path+"uod_val_list.pk",       'wb') as f: pkl.dump(uod_val_list,       f)

---

In [ ]:
with open(analysis_path+"cdf_vals_list_fft.pk",  'rb') as f: cdf_vals_list_fft  = pkl.load(f)
with open(analysis_path+"cdf_vals_list_doip.pk", 'rb') as f: cdf_vals_list_doip = pkl.load(f)
with open(analysis_path+"vmm_log_list.pk",       'rb') as f: vmm_log_list       = pkl.load(f)

with open(analysis_path+"uod_val_list.pk",       'rb') as f: uod_val_list       = pkl.load(f)
with open(analysis_path+"ud_val_list.pk",        'rb') as f: ud_val_list        = pkl.load(f)
with open(analysis_path+"od_val_list.pk",        'rb') as f: od_val_list        = pkl.load(f)

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12,12), dpi=400, sharey=False)
axs = axs.flatten()
for ax in axs: ax.set_box_aspect(1)
# Use axs[3] as a square container for two mini-plots
axs[3].axis("off")
ax3_bot = axs[3].inset_axes([0, 0.00, 1, 0.45])
ax3_top = axs[3].inset_axes([0, 0.55, 1, 0.45], sharex=ax3_bot)
ax3_top.tick_params(labelbottom=False)
ax3s = [ax3_top, ax3_bot]

fig.suptitle(  r"Choosing the origins range: fixed $\delta$ vs fixed $\Delta_{CDF}$" + "\n"
             +  "size="+str(size)
             + r"  |  $R=$"+str(R)+"cMpc/h"
             + r"  |  Lvl="+latex_float(lvl )
             + r"  |  "+cl
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")



ud_on_origin_cdf = []; od_on_origin_cdf = []; uod_on_origin_cdf = [ud_on_origin_cdf, od_on_origin_cdf]
#cdf_fft_ud_vals = []; cdf_fft_od_vals = []; cdf_fft_uod_vals = [cdf_fft_ud_vals, cdf_fft_od_vals]
for Z_index, Z in enumerate(ALL_Zs[sz_indx]):

    Z_fs = ALL_Zs_floatstr[sz_indx][   Z_index]
    
    vmin_log, vmax_log = vmm_log_list[ Z_index]
    x_vals_log = np.linspace(vmin_log, vmax_log, 10**4)
    
    cdf_vals_fft  = cdf_vals_list_fft[ Z_index]
    cdf_vals_doip = cdf_vals_list_doip[Z_index]

    #cdf_fft_ud_vals.append(cdf_vals_fft[np.argmin(np.abs(cdf_vals_doip - ud))])
    #cdf_fft_od_vals.append(cdf_vals_fft[np.argmin(np.abs(cdf_vals_doip - od))])
    ud_on_origin_cdf.append(cdf_vals_doip[np.argmin(np.abs(10**x_vals_log - ud_val_list[Z_index]))])
    od_on_origin_cdf.append(cdf_vals_fft[ np.argmin(np.abs(10**x_vals_log - od_val_list[Z_index]))])

    
    blue_shade = (0, 0.7, 1-Z_index*0.075)
    pink_shade = (1, 0.7, 1-Z_index*0.075)

    lbl0 = None; lbl1 = None; lbl2 = None
    if Z_index in [0, len(ALL_Zs[sz_indx])-1]:
        lbl0 = "z="+Z_fs
        lbl1 = "z="+Z_fs
        lbl2 = "z="+Z_fs
    axs[0].plot(10**x_vals_log, cdf_vals_doip, c=blue_shade, zorder=4, alpha=0.7, lw=0.6, label=lbl0)
    axs[1].plot(cdf_vals_fft,   cdf_vals_doip, c=blue_shade, zorder=4, alpha=0.7, lw=0.6, label=lbl1)
    axs[2].plot(10**x_vals_log, cdf_vals_fft,  c=blue_shade, zorder=4, alpha=0.7, lw=0.6, label=lbl2)


# od & ud - thresholds
axs[0].axhline(od, c="orangered", alpha=0.7, zorder=3, lw=2)
axs[0].scatter(ud_val_list, ud_on_origin_cdf,                      s=7, color="darkorange", label=r"$\delta_{ud} = $"+str(ud))
axs[0].scatter(od_val_list, [od for _ in range(len(od_val_list))], s=7, color="orangered",  label=r"$\delta_{od} = $"+str(od))


axs[1].axvline(ud, c="darkorange", alpha=0.7, zorder=3, lw=2, label=r"$\delta_{ud} = $"+str(ud))
axs[1].axhline(od, c="orangered",  alpha=0.7, zorder=3, lw=2, label=r"$\delta_{od} = $"+str(od))


axs[2].axhline(ud, c="darkorange", alpha=0.7, zorder=3, lw=2)
axs[2].scatter(ud_val_list, [ud for _ in range(len(ud_val_list))], s=7, color="orangered",  label=r"$\delta_{ud} = $"+str(ud))
axs[2].scatter(od_val_list, od_on_origin_cdf,                      s=7, color="darkorange", label=r"$\delta_{od} = $"+str(od))




tks, tkss = set_ticks(10**np.min([_[0] for _ in vmm_log_list]), 10**np.max([_[1] for _ in vmm_log_list]), factor_cut=0.1)
for i0 in range(3):
    axs[i0].grid()
    axs[i0].legend(loc=4)

    if i0 != 1:
        axs[i0].set_xscale("log")
        axs[i0].set_xticks(tks, tkss)


axs[ 0].set_xlabel(r"$\delta+1$")
axs[ 1].set_xlabel(r"$F_{\rm grid}(\delta+1)$")
axs[ 2].set_xlabel(r"$\delta+1$")

axs[ 0].set_ylabel(r"$F_{\rm org}(\delta+1)$")
axs[ 1].set_ylabel(r"$F_{\rm org}(\delta+1)$")
axs[ 2].set_ylabel(r"$F_{\rm grid}(\delta+1)$")








# AXS3
for i0 in range(2):
    ax3s[i0].plot(   ALL_Zs_float[sz_indx], uod_on_origin_cdf[i0], lw=3, zorder=1, c="lightblue")
    ax3s[i0].scatter(ALL_Zs_float[sz_indx], uod_on_origin_cdf[i0], s=10, zorder=2, c="red",
                     label=r"$\delta_{ud}(z) = $"+str(ud)+r" threshold converted"+"\n"+r"from grid CDF to origins CDF" if i0==0 
                     else  r"$\delta_{od}(z) = $"+str(od)+r" threshold converted"+"\n"+r"from origins CDF to grid CDF")

    #tks, tkss, stbl_exp = set_ticks(np.min(cdf_fft_uod_vals[i0]), np.max(cdf_fft_uod_vals[i0]),
    #                                log_lin=False, auto_stable_exponent=True,
    #                                number_of_tks_clean_min=4, number_of_tks_clean_max=5)
    tks, tkss = set_ticks(np.min(uod_on_origin_cdf[i0]), np.max(uod_on_origin_cdf[i0]),
                          log_lin=False, float_g=True, g=4,
                          number_of_tks_clean_min=3, number_of_tks_clean_max=7)
    ax3s[i0].set_yticks(tks, tkss)
    #ax3s[i0].text(0, 0.45+0.03+0.55*i0, rf"$\times 10^{{{stbl_exp}}}$",
    #              transform=ax.transAxes, ha="left", va="top", fontsize=9)
    
    ax3s[i0].legend() 
    ax3s[i0].grid()

ax3s[1].set_xlabel("z")
ax3s[0].set_ylabel(r"$F_{\rm org}(\delta+1)$"); ax3s[1].set_ylabel(r"$F_{\rm grid}(\delta+1)$")






plt.tight_layout()
plt.savefig(plots_path_0+"6.1___CDFs_Zs.png", bbox_inches='tight')
plt.close()

---

In [ ]:
with open(analysis_path+"bins_log_fft.pk",             'rb') as f: bins_log_fft             = pkl.load(f)
with open(analysis_path+"profiles_grid_fft.pk",        'rb') as f: profiles_grid_fft        = pkl.load(f)
with open(analysis_path+"ud_vals.pk",                  'rb') as f: ud_vals                  = pkl.load(f)
with open(analysis_path+"od_vals.pk",                  'rb') as f: od_vals                  = pkl.load(f)
with open(analysis_path+"ud_grid_intersections.pk",    'rb') as f: ud_grid_intersections    = pkl.load(f)
with open(analysis_path+"od_grid_intersections.pk",    'rb') as f: od_grid_intersections    = pkl.load(f)
with open(analysis_path+"ud_origins_intersections.pk", 'rb') as f: ud_origins_intersections = pkl.load(f)
with open(analysis_path+"od_origins_intersections.pk", 'rb') as f: od_origins_intersections = pkl.load(f)

with open(analysis_path+"profiles_origins_fft.pk",     'rb') as f: profiles_origins_fft     = pkl.load(f)
with open(analysis_path+"profiles_grid_fft.pk",        'rb') as f: profiles_grid_fft        = pkl.load(f)

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 8), sharex=True, dpi=400)
axs = axs.flatten()

fig.suptitle(  r"Top: Normalized grid and origins density profiles with their respective intersections with the $\delta_{ud/od}$ thresholds" + "\n"
             + r"Bottom: The respective grid and origins CDFs as a function of density and their respective intersections with the $\delta_{ud/od}$ thresholds" + "\n"
             +       "size=" + str(size)
             + r"  |  $R=$" + str(R) + "cMpc/h"
             + r"  |  Lvl=" + latex_float(lvl)
             + r"  |  " + cl
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")




ud_on_origin_cdf = []; od_on_origin_cdf = []; uod_on_origin_cdf = [ud_on_origin_cdf, od_on_origin_cdf]
for i0 in range(len(ALL_Zs[sz_indx])):

    Z    = ALL_Zs[         sz_indx][i0]
    Z_fs = ALL_Zs_floatstr[sz_indx][i0]

    file_path_Z    = file_path     +"Z___"  +Z                   +"/"
    file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
    file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
    file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"

    

    
    blue_shade = (0, 0.7, 1-i0*0.075)
    lbl0 = None
    if i0 in [0, len(ALL_Zs[sz_indx])-1]: lbl0 = "z="+Z_fs


    ### bottom 2
    vmin_log, vmax_log = vmm_log_list[ i0]
    x_vals_log = np.linspace(vmin_log, vmax_log, 10**4)
    
    cdf_vals_fft  = cdf_vals_list_fft[ i0]; od_on_origin_cdf.append(cdf_vals_fft[ np.argmin(np.abs(10**x_vals_log - od_val_list[i0]))])
    cdf_vals_doip = cdf_vals_list_doip[i0]; ud_on_origin_cdf.append(cdf_vals_doip[np.argmin(np.abs(10**x_vals_log - ud_val_list[i0]))])

    
    axs[2].plot(10**x_vals_log, cdf_vals_fft,  c=blue_shade, alpha=0.7, lw=1.0, zorder=3)
    axs[3].plot(10**x_vals_log, cdf_vals_doip, c=blue_shade, alpha=0.7, lw=1.0, zorder=3)



    
# od & ud - thresholds
ss = 25
#axs[0].scatter(all_ud, intersections_ud_delta,   s=ss, color="darkorange", label=r"$\delta_{ud}$", alpha=0.7, zorder=4)
#axs[0].scatter(all_od, intersections_od_delta,   s=ss, color="orangered",  label=r"$\delta_{od}$", alpha=0.7, zorder=4)

#axs[1].scatter(all_ud, intersections_ud_origins, s=ss, color="darkorange", label=r"$\delta_{ud}$", alpha=0.7, zorder=4)
#axs[1].scatter(all_od, intersections_od_origins, s=ss, color="orangered",  label=r"$\delta_{od}$", alpha=0.7, zorder=4)

axs[2].axhline(ud, c="darkorange", alpha=0.7, zorder=4, lw=2)
axs[2].scatter(ud_val_list, [ud for _ in range(len(ud_val_list))], s=ss, color="darkorange",  label=r"$\delta_{ud} = $"+str(ud))
axs[2].scatter(od_val_list, od_on_origin_cdf,                      s=ss, color="orangered", label=r"$\delta_{od} = $"+str(od))

axs[3].axhline(od, c="orangered", alpha=0.7, zorder=4, lw=2)
axs[3].scatter(od_val_list, [od for _ in range(len(od_val_list))], s=ss, color="orangered",  label=r"$\delta_{od} = $"+str(od))
axs[3].scatter(ud_val_list, ud_on_origin_cdf,                      s=ss, color="darkorange", label=r"$\delta_{ud} = $"+str(ud))


for i0, Z in enumerate(ALL_Zs[sz_indx]):
    lbl = None
    if i0 in [0, len(ALL_Zs[sz_indx])-1]: lbl = "z="+ALL_Zs_floatstr[sz_indx][i0]
    blue_shade = (0, 0.7, 1-i0*0.075)
    axs[0].plot(bins_log_fft, [_/np.sum(profiles_grid_fft[   i0]) for _ in profiles_grid_fft[   i0]], color=blue_shade, lw=1, alpha=0.7, zorder=3, label=lbl)
    axs[1].plot(bins_log_fft, [_/np.sum(profiles_origins_fft[i0]) for _ in profiles_origins_fft[i0]], color=blue_shade, lw=1, alpha=0.7, zorder=3, label=lbl)

lbls = [r"$\delta_{ud}$", r"$\delta_{od}$"]
axs[0].scatter(ud_vals, [ud_grid_intersections[   i0]/np.sum(profiles_grid_fft[   i0]) for i0 in range(len(ALL_Zs[sz_indx]))], color="darkorange", s=5,  alpha=0.7, zorder=4, label=lbls[0])
axs[0].scatter(od_vals, [od_grid_intersections[   i0]/np.sum(profiles_grid_fft[   i0]) for i0 in range(len(ALL_Zs[sz_indx]))], color="orangered",  s=5,  alpha=0.7, zorder=4, label=lbls[1])
axs[1].scatter(ud_vals, [ud_origins_intersections[i0]/np.sum(profiles_origins_fft[i0]) for i0 in range(len(ALL_Zs[sz_indx]))], color="darkorange", s=5,  alpha=0.7, zorder=4, label=lbls[0])
axs[1].scatter(od_vals, [od_origins_intersections[i0]/np.sum(profiles_origins_fft[i0]) for i0 in range(len(ALL_Zs[sz_indx]))], color="orangered",  s=5,  alpha=0.7, zorder=4, label=lbls[1])




for i0 in range(4):
    axs[i0].set_xscale('log')
    axs[i0].set_xlim(bins_log_fft[0], 5)
    axs[i0].grid()

for i0 in range(2,4):
    tks, tkss = set_ticks(bins_log_fft[0], 5)
    axs[i0].set_xticks(tks, tkss)
    axs[i0].set_xlabel(r"$\delta+1$")

axs[1].legend(loc=1)

axs[0].set_ylabel(r"(Normalized) grid density distribution",   labelpad=2)
axs[1].set_ylabel(r"(Normalized) origin density distribution", labelpad=2)
axs[2].set_ylabel(r"$F_{\rm grid}(\delta+1)$")
axs[3].set_ylabel(r"$F_{\rm org}(\delta+1)$")




plt.tight_layout()
plt.savefig(plots_path_0 + "6.1___Grid_and_origin_density_profiles_and_CDFs.png", bbox_inches='tight')
plt.close()

---

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4), sharex=True, dpi=400)
axs = axs.flatten()

fig.suptitle(  r"Top: Normalized grid and origins density profiles with their respective intersections with the $\delta_{ud/od}$ thresholds" + "\n"
             + r"Bottom: The respective grid and origins CDFs as a function of density and their respective intersections with the $\delta_{ud/od}$ thresholds" + "\n"
             +       "size=" + str(size)
             + r"  |  $R=$" + str(R) + "cMpc/h"
             + r"  |  Lvl=" + latex_float(lvl)
             + r"  |  " + cl
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")


axs[0].set_title(r"(Normalized) grid density distribution")
axs[1].set_title(r"(Normalized) origin density distribution")


for i0, Z in enumerate(ALL_Zs[sz_indx]):
    lbl = None
    if i0 in [0, len(ALL_Zs[sz_indx])-1]: lbl = "z="+ALL_Zs_floatstr[sz_indx][i0]
    blue_shade = (0, 0.7, 1-i0*0.075)
    axs[0].plot(bins_log_fft, [_/np.sum(profiles_grid_fft[   i0]) for _ in profiles_grid_fft[   i0]], color=blue_shade, lw=1, alpha=0.7, zorder=3, label=lbl)
    axs[1].plot(bins_log_fft, [_/np.sum(profiles_origins_fft[i0]) for _ in profiles_origins_fft[i0]], color=blue_shade, lw=1, alpha=0.7, zorder=3, label=lbl)

lbls = [r"$\delta_{ud}$", r"$\delta_{od}$"]
axs[0].scatter(ud_vals, [ud_grid_intersections[   i0]/np.sum(profiles_grid_fft[   i0]) for i0 in range(len(ALL_Zs[sz_indx]))], color="darkorange", s=5,  alpha=0.7, zorder=4, label=lbls[0])
axs[0].scatter(od_vals, [od_grid_intersections[   i0]/np.sum(profiles_grid_fft[   i0]) for i0 in range(len(ALL_Zs[sz_indx]))], color="orangered",  s=5,  alpha=0.7, zorder=4, label=lbls[1])
axs[1].scatter(ud_vals, [ud_origins_intersections[i0]/np.sum(profiles_origins_fft[i0]) for i0 in range(len(ALL_Zs[sz_indx]))], color="darkorange", s=5,  alpha=0.7, zorder=4, label=lbls[0])
axs[1].scatter(od_vals, [od_origins_intersections[i0]/np.sum(profiles_origins_fft[i0]) for i0 in range(len(ALL_Zs[sz_indx]))], color="orangered",  s=5,  alpha=0.7, zorder=4, label=lbls[1])




for i0 in range(2):
    axs[i0].set_xlabel(r"$\delta+1$")
    axs[i0].set_xscale('log')
    axs[i0].set_xlim(bins_log_fft[0], 5)
    axs[i0].grid()

axs[1].legend(loc=1)

plt.tight_layout()
plt.savefig(plots_path_0 + "6.1___Grid_and_origin_density_profiles_no_CDFs.png", bbox_inches='tight')
plt.close()

---
---

In [ ]:
sz_indx = 2

size    = BASELINES[sz_indx]["size"]; d_xyz = 75/size
Z       = BASELINES[sz_indx]["Z"]
nnc     = BASELINES[sz_indx]["nnc"]
R       = BASELINES[sz_indx]["R"]
lvl     = BASELINES[sz_indx]["lvl"]
cl      = BASELINES[sz_indx]["cl"]
uod     = BASELINES[sz_indx]["uod"]; ud, od = uod
MK      = BASELINES[sz_indx]["MK"]

file_path      = "../Modified_Data_"+str(size)+"/"
file_path_Z   = file_path    +"Z___"  +Z       +"/"
file_path_NNC = file_path_Z  +"NNC___"+str(nnc)+"/"
file_path_R   = file_path_NNC+"R___"  +str(R)  +"/"
file_path_Lvl = file_path_R  +"Lvl___"+str(lvl)+"/"+cl+"/"
file_path_MK = file_path_Lvl+"D___["+str(ud)+"_"+str(od)+"]/"+MK+"/"

---

In [ ]:
with open(file_path_MK+"fg___100.pk", 'rb') as f: fg_100 = pkl.load(f)
with open(file_path_R+"grid_fft_"+rho_delta+".pk", 'rb') as f: grid_fft = pkl.load(f)

In [ ]:
dr = 1.0

In [ ]:
np.max(fg_100)

In [ ]:
# Re-order the void indices so they start from 0 and no gaps.
fg_100[fg_100 != -2] = np.searchsorted(np.sort(np.unique(fg_100[fg_100 != -2])), fg_100[fg_100 != -2].ravel())
fg_100_sc = fg_100[sc]
fg_100_walls_sc = fg_100_sc == -2
vmin_fg_100 = 0; vmax_fg_100 = np.max(fg_100)

In [ ]:
np.max(fg_100)

In [ ]:
y_density_means1_list = []; x_dist1_list = []

for i9 in range(10):#[_*3 for _ in range(100)]:
    void_0_coords = np.argwhere(fg_100 == i9)
    if len(void_0_coords) != 0:
    
    
        void_0_coords = np.asarray(void_0_coords, dtype=int)
        vals          = grid_fft[void_0_coords[:,0], void_0_coords[:,1], void_0_coords[:,2]]
        
        center = void_0_coords[np.argmin(vals)]
        ci, cj, ck = center
        
        d = void_0_coords - center
        
        d[:,0] = (d[:,0] + size//2)%size - size//2
        d[:,1] = (d[:,1] + size//2)%size - size//2
        d[:,2] = (d[:,2] + size//2)%size - size//2
        
        r = np.sqrt(d[:,0]**2 + d[:,1]**2 + d[:,2]**2)
        
        # Radial bins
        r_max   = np.max(r)
        bins_r  = np.arange(0, r_max + dr, dr)
        bin_idx = np.digitize(r, bins_r) - 1
        
        mask     = (bin_idx >= 0) & (bin_idx < len(bins_r)-1)
        bin_idx  = bin_idx[mask]
        vals_bin = vals[mask]
        
        sum_density  = np.bincount(bin_idx, weights=vals_bin, minlength=len(bins_r)-1)
        counts       = np.bincount(bin_idx, minlength=len(bins_r)-1)
        mask_nonzero = counts > 0
        
        x_dist1 = (bins_r[:-1]+bins_r[1:])/2
        x_dist1 = x_dist1[mask_nonzero]
        
        y_density_means1 = sum_density[mask_nonzero] / counts[mask_nonzero]
        
        y_density_means1_list.append(y_density_means1)
        x_dist1_list.append(         x_dist1)

---

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5), sharey=True, dpi=400)
axs = axs.flatten()

fig.suptitle(  r"Left: spherical approximation single-void radial density profile vs. "+str(len(y_density_means1_list))+" actual WVF found void profiles" + "\n"
             + r"with their respective intersections with the $\delta_{ud/od}$ thresholds" + "\n"
             +       "size=" + str(size)
             + r"  |  $R=$" + str(R) + "cMpc/h"
             + r"  |  Lvl=" + latex_float(lvl)
             + r"  |  " + cl
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")



R_eq = (3 * size**3 / (4 * np.pi))**(1/3)

ud_on_origin_cdf = []; od_on_origin_cdf = []; uod_on_origin_cdf = [ud_on_origin_cdf, od_on_origin_cdf]
for i0 in range(len(ALL_Zs[sz_indx])):

    Z    = ALL_Zs[         sz_indx][i0]
    Z_fs = ALL_Zs_floatstr[sz_indx][i0]

    file_path_Z    = file_path     +"Z___"  +Z                   +"/"
    file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
    file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
    file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"

    
    
    vmin_log, vmax_log = vmm_log_list[ i0]
    x_vals_log = np.linspace(vmin_log, vmax_log, 10**4)
    
    cdf_vals_fft  = cdf_vals_list_fft[ i0]; od_on_origin_cdf.append(cdf_vals_fft[ np.argmin(np.abs(10**x_vals_log - od_val_list[i0]))])
    cdf_vals_doip = cdf_vals_list_doip[i0]; ud_on_origin_cdf.append(cdf_vals_doip[np.argmin(np.abs(10**x_vals_log - ud_val_list[i0]))])

    radial_vals_fft = R_eq * cdf_vals_fft**(1/3)

    blue_shade = (0, 0.7, 1-i0*0.075)
    lbl0 = None
    if i0 in [0, len(ALL_Zs[sz_indx])-1]: lbl0 = "z="+Z_fs
    axs[0].plot(radial_vals_fft, 10**x_vals_log,  c=blue_shade, alpha=0.7, lw=1.0, zorder=3, label=lbl0)



for i9 in range(len(y_density_means1_list)):
    axs[1].plot(x_dist1_list[i9], y_density_means1_list[i9], alpha=0.7, lw=1.0, zorder=3)





axs[0].scatter([R_eq * ud**(1/3) for _ in range(len(ud_val_list))], ud_val_list, s=25, color="darkorange",  label=r"$\delta_{ud} = $"+str(ud))
axs[0].scatter([R_eq *  _**(1/3) for _ in od_on_origin_cdf],        od_val_list, s=25, color="orangered",   label=r"$\delta_{od} = $"+str(od))

for i0 in range(2):
    axs[i0].axhline(ud_val_list[0], c="darkorange", label=r"$\delta_{ud}(z=$"+ALL_Zs_floatstr[sz_indx][0]+"$)$")
    axs[i0].axhline(od_val_list[0], c="orangered" , label=r"$\delta_{od}(z=$"+ALL_Zs_floatstr[sz_indx][0]+"$)$")






axs[0].set_xlim(-10,330)
tks, tkss = set_ticks(0, 330/size*75, log_lin=False, int_if_possible=True, float_asitis_ends=True)
axs[0].set_xticks([_*size/75 for _ in tks], tkss)

axs[1].set_xlim(-10,180)
tks, tkss = set_ticks(0, 180/size*75, log_lin=False, int_if_possible=True, float_asitis_ends=True)
axs[1].set_xticks([_*size/75 for _ in tks], tkss)

for i0 in range(2):
    axs[i0].set_yscale('log')

    vmin_log, vmax_log = vmm_log_list[0]
    x_vals_log = np.linspace(vmin_log, vmax_log, 10**4)
    axs[i0].set_ylim(     10**x_vals_log[0]-0.006, 5)
    tks, tkss = set_ticks(10**x_vals_log[0], 5)
    axs[i0].set_yticks(tks, tkss)
    
    axs[i0].grid()

axs[0].set_ylabel(r"$\delta+1$", labelpad=-30)
axs[0].set_xlabel(r"Radial dinstance from void center [cMpc$/h$]")
axs[1].set_xlabel(r"Radial dinstance from void center [cMpc$/h$]")


axs[0].legend()
plt.tight_layout()
plt.savefig(plots_path_0 + "6.1___Spherical_approx_and_WVF_voids.png", bbox_inches='tight')
plt.close()

---
---
---

## 6.2. Origin Spheres

---

In [ ]:
sz_indx = 2

size    = BASELINES[sz_indx]["size"]; d_xyz = 75/size
Z       = BASELINES[sz_indx]["Z"]
Z_fs    = BASELINES[sz_indx]["Z_floatstr"]
nnc     = BASELINES[sz_indx]["nnc"]
R       = BASELINES[sz_indx]["R"]
lvl     = BASELINES[sz_indx]["lvl"]
cl      = BASELINES[sz_indx]["cl"]
uod     = BASELINES[sz_indx]["uod"]; ud, od = uod
MK      = BASELINES[sz_indx]["MK"]

file_path      = "../Modified_Data_"+str(size)+"/"

file_path_Z    = file_path     +"Z___"  +Z                   +"/"
file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"
file_path_Duod = file_path_Lvl +"D___[" +str(ud)+"_"+str(od)+"]/"
file_path_MK   = file_path_Duod+MK                           +"/"

In [ ]:
X, Y = np.meshgrid(np.linspace(0, 75, size), np.linspace(0, 75, size)); Y = Y[::-1]

---

In [ ]:
with open(file_path_MK+"fg___100.pk", 'rb') as f: fg_100_MK2 = pkl.load(f)

fg_100_MK2_walls = np.ma.masked_where(fg_100_MK2 > -2, fg_100_MK2)

---

In [ ]:
with open(file_path_Lvl+"dict_origins_isolated.pk", 'rb') as f: origins_isolated = pkl.load(f)
with open(file_path_Lvl+"dict_origins_pairs.pk",    'rb') as f: origins_pairs    = pkl.load(f)

In [ ]:
radius = 10

print(round(75/size*radius,2), "Mpc/h")

In [ ]:
grid_origins_isolated = np.zeros((size, size, size))
grid_origins_pairs    = np.zeros((size, size, size))

for i in nb.prange(len(origins_isolated)):
    for j in origins_isolated[i]:
        
        for i0 in range(0-radius,1+radius):
            for i1 in range(0-radius,1+radius):
                for i2 in range(0-radius,1+radius):
                    
                    if np.sqrt(i0**2 + i1**2 + i2**2) <= radius:
                        grid_origins_isolated[(j[0]+i0)%size][(j[1]+i1)%size][(j[2]+i2)%size] = 1+radius+i0

for i in nb.prange(len(origins_pairs)):
    for j in origins_pairs[i]:
        for k in j:
            
            for i0 in range(0-radius,1+radius):
                for i1 in range(0-radius,1+radius):
                    for i2 in range(0-radius,1+radius):
                        
                        if np.sqrt(i0**2 + i1**2 + i2**2) <= radius:
                            grid_origins_pairs[(k[0]+i0)%size][(k[1]+i1)%size][(k[2]+i2)%size] = 1+radius+i0

In [ ]:
grid_origins_isolated_mask = np.ma.masked_where(grid_origins_isolated == 0, grid_origins_isolated)
grid_origins_pairs_mask    = np.ma.masked_where(grid_origins_pairs    == 0, grid_origins_pairs)

In [ ]:
grid_origins_isolated_mask -= 1+radius
grid_origins_pairs_mask    -= 1+radius

In [ ]:
grid_origins_isolated_mask *= 75/size
grid_origins_pairs_mask    *= 75/size

---

In [ ]:
colors1 = ['#FFB6C1', '#FFFFE0', '#FFE4B5']  # Light pink -> pale yellow -> light orange
cmap1 = LinearSegmentedColormap.from_list('bright_pink_yellow_orange', colors1, N=256)

colors2 = ['#00008B', '#4B0082', '#8B0000']  # Dark blue -> dark purple -> dark red
cmap2 = LinearSegmentedColormap.from_list('dark_blue_purple_red', colors2, N=256)

---

In [ ]:
fig, axs = plt.subplots(1, figsize=(8,5), dpi=400)

fig.suptitle(r"Isolated and pair origins as spheres into the plane compared to the WFV walls." + "\n"
             +  r"$yz$-slice [cMpc/h]"
                 +  "  |  size="+str(size)
                 +  "  |  Z="+Z_fs
                 + r"  |  Lvl="+latex_float(lvl)
                 + r"  |  "+cl
                 + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")


axs.pcolormesh(X, Y, fg_100_MK2_walls[sc], cmap=ListedColormap(['lime']))


imm_i = axs.pcolormesh(X, Y, grid_origins_isolated_mask[sc], cmap=cmap1, vmin=-radius*75/size, vmax=radius*75/size)
imm_p = axs.pcolormesh(X, Y, grid_origins_pairs_mask[   sc], cmap=cmap2, vmin=-radius*75/size, vmax=radius*75/size)


divider = make_axes_locatable(axs)
cax = divider.append_axes('right', size='5%', pad=0.95)
cbar = plt.colorbar(imm_i, cax=cax, orientation='vertical')
tks, tkss = set_ticks(-radius*75/size, radius*75/size, log_lin=False, int_if_possible=True)
cbar.ax.set_yticks(tks, tkss)
cbar.set_label("Distance from the plane to the\ncenter of isolated origins [cMpc/h]", rotation=270, labelpad=10)

divider = make_axes_locatable(axs)
cax = divider.append_axes('left', size='5%', pad=0.35)
cbar = plt.colorbar(imm_p, cax=cax, orientation='vertical')
tks, tkss = set_ticks(-radius*75/size, radius*75/size, log_lin=False, int_if_possible=True)
cbar.ax.set_yticks(tks, tkss)
cbar.ax.yaxis.set_ticks_position('left')
cbar.set_label("Distance from the plane to the\ncenter of origin pairs [cMpc/h]", rotation=90, labelpad=-55)

axs.set_xticks(range_75_5, range_75_5_tkss); axs.set_yticks(range_75_5, range_75_5_tkss)

axs.set_aspect('equal')

plt.tight_layout()
plt.savefig(plots_path_0+"6.2___Origins_spheres.png", bbox_inches='tight')
plt.close()

---

In [ ]:
del grid_origins_isolated, grid_origins_pairs, origins_isolated, origins_pairs; gc.collect()

---
---
---

## 6.3. $z=0.0$ CDFs

---

In [ ]:
sz_indx = 2

size    = BASELINES[sz_indx]["size"]; d_xyz = 75/size
Z       = BASELINES[sz_indx]["Z"]
nnc     = BASELINES[sz_indx]["nnc"]
R       = BASELINES[sz_indx]["R"]
lvl     = BASELINES[sz_indx]["lvl"]
cl      = BASELINES[sz_indx]["cl"]
uod     = BASELINES[sz_indx]["uod"]; ud, od = uod
MK      = BASELINES[sz_indx]["MK"]

file_path      = "../Modified_Data_"+str(size)+"/"

file_path_Z    = file_path     +"Z___"  +Z                   +"/"
file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"

In [ ]:
X, Y = np.meshgrid(np.linspace(0, 75, size), np.linspace(0, 75, size)); Y = Y[::-1]

---

In [ ]:
uod

In [ ]:
len(ALL_uods[sz_indx][0][2][0][2][0]), ALL_uods[sz_indx][0][2][0][2][0]

In [ ]:
uod_other = copy.deepcopy(ALL_uods[sz_indx][0][2][0][2][0])
uod_other.remove(uod)
len(uod_other), uod_other

In [ ]:
uod_other = [uod for _ in range(7)]

---

In [ ]:
with open(file_path_R+"grid_fft_"+rho_delta+".pk", 'rb') as f: grid_fft = pkl.load(f)
grid_fft_sc = grid_fft[sc]
vmin_fft = np.min(grid_fft); vmax_fft = np.max(grid_fft)

In [ ]:
fig, axs = plt.subplots(4, 2, dpi=400, figsize=(16, 18.4))
axs = axs.flatten()

fig.suptitle(r"Walls & Density $\Delta_{CDF} comparison$" + "\n"
             + r"$yz$-slice [cMpc/h]"
             +  "  |  size="+str(size)
             + r"  |  $R=$"+str(R)+"cMpc/h"
             + r"  |  Lvl="+latex_float(lvl)
             + r"  |  "+cl
             +  "  |  "+MK
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")


for i0 in range(8):
    
    imm = axs[i0].pcolormesh(X, Y, grid_fft_sc, norm=LogNorm(vmin=vmin_fft, vmax=vmax_fft), cmap='RdBu_r', alpha=1, zorder=1)
    axs[i0].set_aspect('equal')

    divider = make_axes_locatable(axs[i0])
    cax = divider.append_axes('right', size='5%', pad=0.95)
    cbar = plt.colorbar(imm, cax=cax, orientation='vertical')
    tks, tkss = set_ticks(vmin_fft, vmax_fft)
    cbar.ax.set_yticks(tks, tkss)
    cbar.set_label(r'$\delta+1$', rotation=270, labelpad=-15)






    #ud, od = uod
    if i0 == 0: ud, od = uod
    else:       ud, od = uod_other[i0-1]
    
    axs[i0].set_title(r"$\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")
    
    file_path_Duod = file_path_Lvl +"D___[" +str(ud)+"_"+str(od)+"]/"
    file_path_MK   = file_path_Duod+MK                           +"/"
    
    # Plot walls
    with open(file_path_MK+"fg___100.pk", 'rb') as f: fg_100 = pkl.load(f)

    # Re-order the void indices so they start from 0 and no gaps.
    fg_100[fg_100 != -2] = np.searchsorted(np.sort(np.unique(fg_100[fg_100 != -2])), fg_100[fg_100 != -2].ravel())
    fg_100_sc = fg_100[sc]
    fg_100_walls_sc = fg_100_sc == -2
    vmin_fg_100 = 0; vmax_fg_100 = np.max(fg_100)
    
    imm = axs[i0].imshow(fg_100_sc,                                       cmap='binary', extent=[0.0, 75, 0.0, 75], alpha=0.7, zorder=2, vmin=vmin_fg_100, vmax=vmax_fg_100)
    axs[      i0].imshow(np.ma.masked_where(~fg_100_walls_sc, fg_100_sc), cmap='winter', extent=[0.0, 75, 0.0, 75], alpha=0.7, zorder=2)


    divider = make_axes_locatable(axs[i0])
    cax = divider.append_axes('left', size='5%', pad=0.45)
    cbar = plt.colorbar(imm, cax=cax, orientation='vertical')
    cbar.ax.yaxis.set_ticks_position('left')
    tks, tkss = set_ticks(vmin_fg_100, vmax_fg_100, log_lin=False, int_if_possible=True)
    cbar.ax.set_yticks(tks, tkss)
    cbar.set_label(r'Void index (reordered)', rotation=90, labelpad=-55)



    axs[i0].set_xticks(range_75_5, range_75_5_tkss)
    axs[i0].set_yticks(range_75_5, range_75_5_tkss)
    
plt.tight_layout()
plt.savefig(plots_path_0+"6.3___Z_"+Z+"___CDFs.png", bbox_inches='tight')
plt.close()

---
---
---
---
---
---
---
---
---
---

# 7. BUILD-UP

---
---
---

## 7.1 Build-up Frames

---

In [ ]:
sz_indx = 2

size    = BASELINES[sz_indx]["size"]; d_xyz = 75/size
Z       = BASELINES[sz_indx]["Z"]
Z_fs    = BASELINES[sz_indx]["Z_floatstr"]
nnc     = BASELINES[sz_indx]["nnc"]
R       = BASELINES[sz_indx]["R"]
lvl     = BASELINES[sz_indx]["lvl"]
cl      = BASELINES[sz_indx]["cl"]
uod     = BASELINES[sz_indx]["uod"]; ud, od = uod

file_path      = "../Modified_Data_"+str(size)+"/"

file_path_Z    = file_path     +"Z___"  +Z                   +"/"
file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"
file_path_Duod = file_path_Lvl +"D___[" +str(ud)+"_"+str(od)+"]/"
file_path_MK1   = file_path_Duod+"MK1"                           +"/"
file_path_MK2   = file_path_Duod+"MK2"                           +"/"

---

In [ ]:
with open(file_path_MK1+"fg___100.pk", 'rb') as f: fg_MK1 = pkl.load(f)
with open(file_path_MK2+"fg___100.pk", 'rb') as f: fg_MK2 = pkl.load(f)

In [ ]:
fg_MK1[fg_MK1 != -2] = 0; fg_MK1[fg_MK1 == -2] = 1
fg_MK2[fg_MK2 != -2] = 0; fg_MK2[fg_MK2 == -2] = 1

In [ ]:
aa0 = []; aa1 = []; aa2 = []
for i0 in range(size):
    aa0.append(np.sum(np.abs(fg_MK2[i0, :, :] - fg_MK1[i0, :, :])))
    aa1.append(np.sum(np.abs(fg_MK2[:, i0, :] - fg_MK1[:, i0, :])))
    aa2.append(np.sum(np.abs(fg_MK2[:, :, i0] - fg_MK1[:, :, i0])))

---

In [ ]:
fig = plt.figure(dpi=150, figsize=(12,5))

fig.suptitle(  r"x/y/z slices: numbers of different walls in each slice between MK1 and MK2" + "\n"
             +       "size="+str(size)
             +  "  |  z="+Z_fs
             + r"  |  $R=$"+str(R)+"cMpc/h"
             + r"  |  Lvl="+latex_float(lvl)
             + r"  |  "+cl
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")

plt.plot(np.linspace(0,size-1, size), aa0, label="x - slice", alpha=0.8, c="green")
plt.plot(np.linspace(0,size-1, size), aa1, label="y - slice", alpha=0.8, c="red")
plt.plot(np.linspace(0,size-1, size), aa2, label="z - slice", alpha=0.8, c="blue")

tks, tkss = set_ticks(0, size, log_lin=False, int_if_possible=True)
plt.xticks(tks, tkss)
tks, tkss = set_ticks(0, np.max([np.max(aa0), np.max(aa1), np.max(aa2)]), log_lin=False, int_if_possible=True)
plt.yticks(tks, tkss)

plt.xlabel("Cells along the x/y/z axis")
plt.ylabel("Wall counts")

plt.legend()
plt.grid()
plt.savefig(plots_path_0+"7.1___Max_walls_change.png", bbox_inches='tight')
plt.close()

From analyzing the frames, the difference at the max value is due to a new wall appearing in that slice and there is no reason to doubt one method over the other.

---
---
---

## 7.2 Build-up Plot Each Percent

---

In [ ]:
sz_indx = 2

size    = BASELINES[sz_indx]["size"]; d_xyz = 75/size
Z       = BASELINES[sz_indx]["Z"]
Z_fs    = BASELINES[sz_indx]["Z_floatstr"]
nnc     = BASELINES[sz_indx]["nnc"]
R       = BASELINES[sz_indx]["R"]
lvl     = BASELINES[sz_indx]["lvl"]
cl      = BASELINES[sz_indx]["cl"]
uod     = BASELINES[sz_indx]["uod"]; ud, od = uod

file_path      = "../Modified_Data_"+str(size)+"/"

file_path_Z    = file_path     +"Z___"  +Z                   +"/"
file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"
file_path_Duod = file_path_Lvl +"D___[" +str(ud)+"_"+str(od)+"]/"
file_path_MK1   = file_path_Duod+"MK1"                           +"/"
file_path_MK2   = file_path_Duod+"MK2"                           +"/"

In [ ]:
X, Y = np.meshgrid(np.linspace(0, 75, size), np.linspace(0, 75, size)); Y = Y[::-1]

---

In [ ]:
path_buildup = plots_path_0+"7.2___buildup__Z="+str(Z)+"__["+str(ud)+", "+str(od)+"]/"
if not os.path.exists(path_buildup): os.makedirs(path_buildup)

---

In [ ]:
with open(file_path_R+"grid_fft_"+rho_delta+".pk", 'rb') as f: grid_fft = pkl.load(f)
grid_fft_sc = grid_fft[sc]
vmin_fft = np.min(grid_fft); vmax_fft = np.max(grid_fft)

---

In [ ]:
green_cmap = mcolors.LinearSegmentedColormap.from_list('custom_green', [(0, 1, 0), (0, 1, 0)])

In [ ]:
total_walls_diff = []; total_walls_MK2 = []
for perc in range(101):

    fig, axs = plt.subplots(1, 2, figsize=(16,6.4), dpi=400)

    fig.suptitle(  r"Voids Buildup: MK1 (left) vs. MK2 (right)" + "\n"
                 + r"$yz$-slice [cMpc/h]"
                 +  "  |  size="+str(size)
                 +  "  |  z="+Z_fs
                 + r"  |  $R=$"+str(R)+"cMpc/h"
                 + r"  |  Lvl="+latex_float(lvl)
                 + r"  |  "+cl
                 + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]"
                 + r"  |  Completion: "+str(perc)+"%")
    
    for i00 in range(2):
        imm = axs[i00].pcolormesh(X, Y, grid_fft_sc, norm=LogNorm(vmin=vmin_fft, vmax=vmax_fft), cmap='RdBu_r', alpha=1, zorder=1)
        axs[i00].set_aspect('equal')

        if i00 == 1:
            divider = make_axes_locatable(axs[i00])
            cax = divider.append_axes('right', size='5%', pad=0.95)
            if i00 == 0:
                cax.set_xticks([], []); cax.set_yticks([], []); cax.axis('off')
            else:
                cbar = plt.colorbar(imm, cax=cax, orientation='vertical')
                tks, tkss = set_ticks(vmin_fft, vmax_fft)
                cbar.ax.set_yticks(tks, tkss)
                cbar.set_label(r'$\delta+1$', rotation=270, labelpad=-15)
    
        axs[i00].set_xticks(range_75_5, range_75_5_tkss)
        axs[i00].set_yticks(range_75_5, range_75_5_tkss)
    

    if perc == 0:
        with open(file_path_Lvl+"fg___0_MK1.pk",         'rb') as f: fg_i_MK1 = pkl.load(f)
        with open(file_path_Lvl+"fg___0_MK2.pk",         'rb') as f: fg_i_MK2 = pkl.load(f)
    else:
        with open(file_path_MK1+"fg___"+str(perc)+".pk", 'rb') as f: fg_i_MK1 = pkl.load(f)
        with open(file_path_MK2+"fg___"+str(perc)+".pk", 'rb') as f: fg_i_MK2 = pkl.load(f)
    
    # Re-order the void indices so they start from 0 and no gaps.
    fg_i_MK1[fg_i_MK1 >= 0] = np.searchsorted(np.sort(np.unique(fg_i_MK1[fg_i_MK1 >= 0])), fg_i_MK1[fg_i_MK1 >= 0].ravel())
    fg_i_MK2[fg_i_MK2 >= 0] = np.searchsorted(np.sort(np.unique(fg_i_MK2[fg_i_MK2 >= 0])), fg_i_MK2[fg_i_MK2 >= 0].ravel())
    
    fg_i_MK1_walls = fg_i_MK1 == -2; fg_i_MK1_walls_sc = fg_i_MK1_walls[sc]
    fg_i_MK2_walls = fg_i_MK2 == -2; fg_i_MK2_walls_sc = fg_i_MK2_walls[sc]
    
    total_walls_diff.append(np.sum(np.logical_xor(fg_i_MK1_walls, fg_i_MK2_walls)))
    total_walls_MK2.append(np.sum(fg_i_MK2_walls))
    
    vmin_i_MK = 0; vmax_i_MK1 = np.max(fg_i_MK1); vmax_i_MK2 = np.max(fg_i_MK2)
    fg_i_MK1_sc = fg_i_MK1[sc]; fg_i_MK2_sc = fg_i_MK2[sc]
    
    fg_i_MK12_sc       = [fg_i_MK1_sc,       fg_i_MK2_sc]
    fg_i_MK12_walls_sc = [fg_i_MK1_walls_sc, fg_i_MK2_walls_sc]
    vmax_i_MK12        = [vmax_i_MK1,        vmax_i_MK2]
    
    for i00 in range(2):
        imm = axs[i00].imshow(np.ma.masked_where(fg_i_MK12_sc[i00] == -1, fg_i_MK12_sc[i00]), vmin=vmin_i_MK, vmax=vmax_i_MK12[i00], cmap='spring', extent=[0.0, 75, 0.0, 75], zorder=2)
        axs[      i00].imshow(np.ma.masked_where(~fg_i_MK12_walls_sc[i00], np.ma.masked_where(~fg_i_MK12_walls_sc[i00], fg_i_MK12_sc[i00])), cmap=green_cmap, extent=[0.0, 75, 0.0, 75], zorder=2)

        divider = make_axes_locatable(axs[i00])
        cax = divider.append_axes('left', size='5%', pad=0.45)
        cbar = plt.colorbar(imm, cax=cax, orientation='vertical')
        cbar.ax.yaxis.set_ticks_position('left')
        tks, tkss = set_ticks(vmin_i_MK, vmax_i_MK12[i00], log_lin=False, int_if_possible=True, factor_cut=1/19)
        cbar.ax.set_yticks(tks, tkss)
        cbar.set_label(r'Void index (reordered)', rotation=90, labelpad=-70)
    
    plt.tight_layout()
    plt.savefig(path_buildup+"perc_"+str(perc)+".png", bbox_inches='tight')
    plt.close()

In [ ]:
total_walls_diff = np.array(total_walls_diff)
total_walls_MK2  = np.array(total_walls_MK2)

In [ ]:
with open(analysis_path+"total_walls_diff.pk", 'wb') as f: pkl.dump(total_walls_diff, f)
with open(analysis_path+"total_walls_MK2.pk",  'wb') as f: pkl.dump(total_walls_MK2,  f)

---

In [ ]:
del fg_i_MK1_walls, fg_i_MK2_walls, X; gc.collect()

---
---
---

### Now turn them into a video

In [ ]:
INPUT_FOLDER = Path("../Plots/7.2___buildup__Z=00__[0.1, 0.02]/")

PREFIX    = "perc_"
EXTENSION = ".png"

OUTPUT_NAME   = "perc__buildup_video.mp4"
OUTPUT_FOLDER = None   # None = same as INPUT_FOLDER

DISPLAY_SECONDS = 0.2
VIDEO_FPS       = int(1/DISPLAY_SECONDS)   # I mean, you could always use sth else...

CODEC = "avc1"   # I can see this one with QuickTime Player but feel free to change it

In [ ]:
make_video()

---
---
---

In [ ]:
with open(analysis_path+"total_walls_diff.pk", 'rb') as f: total_walls_diff = pkl.load(f)
with open(analysis_path+"total_walls_MK2.pk",  'rb') as f: total_walls_MK2  = pkl.load(f)

In [ ]:
perc_walls = []; x_vals = []
for i0 in range(101):
    if total_walls_MK2[i0] != 0:
        x_vals.append(i0)
        perc_walls.append(total_walls_diff[i0]/total_walls_MK2[i0]*100)

---

In [ ]:
fig, axs = plt.subplots(figsize=(12,5), dpi=400)

fig.suptitle(  r"Percentage of different walls: MK1 vs. MK2" + "\n"
             +       "size="+str(size)
             +  "  |  z="+Z_fs
             + r"  |  $R=$"+str(R)+"cMpc/h"
             + r"  |  Lvl="+latex_float(lvl)
             + r"  |  "+cl
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")



axs.axvline(100,            c='black', lw=5,   alpha=0.4)
axs.axhline(perc_walls[-1], c='blue',  lw=0.8, alpha=0.4)
plt.annotate("Thinned walls MK1: "+str(round(perc_walls[-1],2))+'%',xy=(77.5, perc_walls[-1]+0.5), fontsize=7, color="blue")

axs.plot(   x_vals, perc_walls, c='red', lw=0.8)
axs.scatter(x_vals, perc_walls, c='red', s=1)

#plt.annotate(str(round(perc_walls[-1],2))+'%',xy=(95, perc_walls[-2]-0.5), fontsize=7)




tkss = [str(_*10) for _ in range(101)]
tkss[-1] = "100\nthin"

axs.set_ylim(0, 1.1*np.max(perc_walls))
axs.set_xlim(-1,101)

axs.set_xlabel("Completion buildup (percentage)")
axs.set_ylabel("Difference (percentage)")

plt.grid()
plt.savefig(plots_path_0+"7.2___Percentages_evolution.png", bbox_inches='tight')
plt.close()

---
---
---
---
---
---
---
---
---
---

# 8. DENSITIES & VOLUMES

---
---
---

In [ ]:
sz_indx = 2

size    = BASELINES[sz_indx]["size"]; d_xyz = 75/size
uod     = BASELINES[sz_indx]["uod"]; ud, od = uod

In [ ]:
if not os.path.exists(plots_path_0+"8.1___"+str(size)+"_Zs"):                              os.makedirs(plots_path_0+"8.1___"+str(size)+"_Zs")
if not os.path.exists(plots_path_0+"8.1___"+str(size)+"_Zs/"+str(ud)+"_"+str(od)+"/ALL"):  os.makedirs(plots_path_0+"8.1___"+str(size)+"_Zs/"+str(ud)+"_"+str(od)+"/ALL")

---
---
---

## 8.1 Walls evolution

---

In [ ]:
sz_indx = 2

size    = BASELINES[sz_indx]["size"]; d_xyz = 75/size
Z       = BASELINES[sz_indx]["Z"]
Z_fs    = BASELINES[sz_indx]["Z_floatstr"]
nnc     = BASELINES[sz_indx]["nnc"]
R       = BASELINES[sz_indx]["R"]
lvl     = BASELINES[sz_indx]["lvl"]
cl      = BASELINES[sz_indx]["cl"]
MK      = BASELINES[sz_indx]["MK"]

file_path      = "../Modified_Data_"+str(size)+"/"

In [ ]:
X, Y = np.meshgrid(np.linspace(0, 75, size), np.linspace(0, 75, size)); Y = Y[::-1]

---

In [ ]:
ALL_uods[sz_indx][0][2][0][2][0]

---

In [ ]:
density_voids = []; volume_voids = []; dmedian_voids = []
for i000 in range(2,3):

    density_voids.append([]); dmedian_voids.append([]); volume_voids.append([])

    ud, od = ALL_uods[sz_indx][0][2][0][2][0][i000]
 
    for i0 in range(len(ALL_Zs[sz_indx])):
        
        Z    = ALL_Zs[         sz_indx][i0]; print(Z)
        Z_fs = ALL_Zs_floatstr[sz_indx][i0]

        file_path_Z    = file_path     +"Z___"  +Z                   +"/"
        file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
        file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
        file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"
        file_path_Duod = file_path_Lvl +"D___[" +str(ud)+"_"+str(od)+"]/"
        file_path_MK   = file_path_Duod+MK                           +"/"
        
        
        
        with open(file_path_R+"grid_fft_"+rho_delta+".pk", 'rb') as f: grid_fft = pkl.load(f)
        grid_fft_sc = grid_fft[sc]
        vmin_fft = np.min(grid_fft); vmax_fft = np.max(grid_fft)

        
        with open(file_path_MK+"fg___100.pk", 'rb') as f: fg_100 = pkl.load(f)
        # Re-order the void indices so they start from 0 and no gaps.
        fg_100[fg_100 != -2] = np.searchsorted(np.sort(np.unique(fg_100[fg_100 != -2])), fg_100[fg_100 != -2].ravel())


        
        no_voids = np.max(fg_100)+1   # +1 for the index 0 void
        density = np.zeros(no_voids+1); dmedian = np.zeros(no_voids+1); volume = np.zeros(no_voids+1)
        
        d_xyz3 = d_xyz**3
        for iii in range(no_voids):
            grid_iii = grid_fft[fg_100 == iii]
            
            density[iii] = np.mean(grid_iii)
            dmedian[iii] = np.median(grid_iii)
            volume[ iii] = len(grid_iii)*d_xyz3

        grid_iii = grid_fft[fg_100 == -2]
        density[-1] = np.mean(grid_iii)
        dmedian[-1] = np.median(grid_iii)
        volume[ -1] = len(grid_iii)*d_xyz3
        
        density_voids[-1].append(density); dmedian_voids[-1].append(dmedian); volume_voids[-1].append(volume)

In [ ]:
with open(analysis_path+"density_voids.pk", 'wb') as f: pkl.dump(density_voids, f)
with open(analysis_path+"dmedian_voids.pk", 'wb') as f: pkl.dump(dmedian_voids, f)
with open(analysis_path+"volume_voids.pk",  'wb') as f: pkl.dump(volume_voids,  f)

---

In [ ]:
for i000 in range(2,3):

    ud, od = ALL_uods[sz_indx][0][2][0][2][0][i000]
    #uod = BASELINES[sz_indx]["uod"]; ud, od = uod
    if not os.path.exists(plots_path_0+"8.1___"+str(size)+"_Zs/"+str(ud)+"_"+str(od)+"/"):  os.makedirs(plots_path_0+"8.1___"+str(size)+"_Zs/"+str(ud)+"_"+str(od)+"/")
    
    for i00 in range(2):
    
        ct_0 = 8
        if i00 == 1: ct_0 = 6
        
        
        fig, axs = plt.subplots(int(ct_0/2), 2, dpi=400, figsize=(16, 2.3*ct_0))
        
        fig.suptitle(r"Walls & Density Z-evolution" + "\n"
                     + r"$yz$-slice [cMpc/h]"
                     +  "  |  size="+str(size)
                     + r"  |  $R=$"+str(R)+"cMpc/h"
                     + r"  |  Lvl="+latex_float(lvl)
                     + r"  |  "+cl
                     +  "  |  "+MK
                     + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]" + "\n")
        
        axs = axs.flatten()
    
        i0_axs = -1
        for i0 in range(8*i00,8*i00+ct_0):
            i0_axs += 1
            
            
            Z    = ALL_Zs[         sz_indx][i0]
            Z_fs = ALL_Zs_floatstr[sz_indx][i0]
    
            axs[i0_axs].set_title("z="+Z_fs)

            file_path_Z    = file_path     +"Z___"  +Z                   +"/"
            file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
            file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
            file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"
            file_path_Duod = file_path_Lvl +"D___[" +str(ud)+"_"+str(od)+"]/"
            file_path_MK   = file_path_Duod+MK                           +"/"
            
            
            
            with open(file_path_R+"grid_fft_"+rho_delta+".pk", 'rb') as f: grid_fft = pkl.load(f)
            grid_fft_sc = grid_fft[sc]
            vmin_fft = np.min(grid_fft); vmax_fft = np.max(grid_fft)
    
            imm = axs[i0_axs].pcolormesh(X, Y, grid_fft_sc, norm=LogNorm(vmin=vmin_fft, vmax=vmax_fft), cmap='RdBu_r', alpha=1, zorder=1)
            axs[i0_axs].set_aspect('equal')
    
            divider = make_axes_locatable(axs[i0_axs])
            cax = divider.append_axes('right', size='5%', pad=0.95)
            cbar = plt.colorbar(imm, cax=cax, orientation='vertical')
            tks, tkss = set_ticks(vmin_fft, vmax_fft)
            cbar.ax.set_yticks(tks, tkss)
            cbar.set_label(r'$\delta+1$', rotation=270, labelpad=-15)
    
    
            # Plot walls
            with open(file_path_MK+"fg___100.pk", 'rb') as f: fg_100 = pkl.load(f)
        
            # Re-order the void indices so they start from 0 and no gaps.
            fg_100[fg_100 != -2] = np.searchsorted(np.sort(np.unique(fg_100[fg_100 != -2])), fg_100[fg_100 != -2].ravel())
            fg_100_sc = fg_100[sc]
            fg_100_walls_sc = fg_100_sc == -2
            vmin_fg_100 = 0; vmax_fg_100 = np.max(fg_100)
            
            imm = axs[i0_axs].imshow(fg_100_sc,                                       cmap='binary', extent=[0.0, 75, 0.0, 75], alpha=0.7, zorder=2, vmin=vmin_fg_100, vmax=vmax_fg_100)
            axs[      i0_axs].imshow(np.ma.masked_where(~fg_100_walls_sc, fg_100_sc), cmap='winter', extent=[0.0, 75, 0.0, 75], alpha=0.7, zorder=2)
    
    
            divider = make_axes_locatable(axs[i0_axs])
            cax = divider.append_axes('left', size='5%', pad=0.45)
            cbar = plt.colorbar(imm, cax=cax, orientation='vertical')
            cbar.ax.yaxis.set_ticks_position('left')
            tks, tkss = set_ticks(vmin_fg_100, vmax_fg_100, log_lin=False, int_if_possible=True)
            cbar.ax.set_yticks(tks, tkss)
            cbar.set_label(r'Void index (reordered)', rotation=90, labelpad=-55)
            
            axs[i0_axs].set_xticks(range_75_5, range_75_5_tkss)
            axs[i0_axs].set_yticks(range_75_5, range_75_5_tkss)



            fig1, axs1 = plt.subplots(dpi=400, figsize=(8,6))
            
            fig1.suptitle(r"Walls & Density Z-evolution" + "\n"
                          + r"$yz$-slice [cMpc/h]"
                          +  "  |  size="+str(size)
                          +  "  |  Z="+Z_fs
                          + r"  |  $R=$"+str(R)+"cMpc/h"
                          + r"  |  Lvl="+latex_float(lvl)
                          + r"  |  "+cl
                          +  "  |  "+MK
                          + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]" + "\n")
            
            imm1 = axs1.pcolormesh(X, Y, grid_fft_sc, norm=LogNorm(vmin=vmin_fft, vmax=vmax_fft), cmap='RdBu_r', alpha=1, zorder=1)
            axs1.set_aspect('equal')
            
            divider = make_axes_locatable(axs1)
            cax = divider.append_axes('right', size='5%', pad=1.15)
            cbar = plt.colorbar(imm1, cax=cax, orientation='vertical')
            tks, tkss = set_ticks(vmin_fft, vmax_fft)
            cbar.ax.set_yticks(tks, tkss)
            cbar.set_label(r'$\delta+1$', rotation=270, labelpad=-15)
            
            imm1 = axs1.imshow(fg_100_sc,                                cmap='binary', extent=[0.0, 75, 0.0, 75], alpha=0.7, zorder=2, vmin=vmin_fg_100, vmax=vmax_fg_100)
            axs1.imshow(np.ma.masked_where(~fg_100_walls_sc, fg_100_sc), cmap='winter', extent=[0.0, 75, 0.0, 75], alpha=0.7, zorder=2)
            
            divider = make_axes_locatable(axs1)
            cax = divider.append_axes('left', size='5%', pad=0.45)
            cbar = plt.colorbar(imm1, cax=cax, orientation='vertical')
            cbar.ax.yaxis.set_ticks_position('left')
            tks, tkss = set_ticks(vmin_fg_100, vmax_fg_100, log_lin=False, int_if_possible=True)
            cbar.ax.set_yticks(tks, tkss)
            cbar.set_label(r'Void index (reordered)', rotation=90, labelpad=-55)
            
            axs1.set_xticks(range_75_5, range_75_5_tkss)
            axs1.set_yticks(range_75_5, range_75_5_tkss)
            
            fig1.tight_layout()
            fig1.savefig(plots_path_0+"8.1___"+str(size)+"_Zs/"+str(ud)+"_"+str(od)+"/Frame__Z_"+Z_fs+".png", bbox_inches='tight')
            plt.close(fig1)
    

        
        plt.tight_layout()
        plt.savefig(plots_path_0+"8.1___"+str(size)+"_Zs/"+str(ud)+"_"+str(od)+"/ALL/Frame__Z_ALL_"+str(i00)+".png", bbox_inches='tight')
        plt.close()

In [ ]:
INPUT_FOLDER = Path(plots_path_0+"8.1___"+str(size)+"_Zs/"+str(ud)+"_"+str(od)+"/")

PREFIX    = None
EXTENSION = ".png"

OUTPUT_NAME   = "Z_walls_and_grid_video.mp4"
OUTPUT_FOLDER = None   # None = same as INPUT_FOLDER

DISPLAY_SECONDS = 0.5
VIDEO_FPS       = int(1/DISPLAY_SECONDS)   # I mean, you could always use sth else...

CODEC = "avc1"   # I can see this one with QuickTime Player but feel free to change it

make_video(reverse_ORDER = True)

---
---
---

## 8.2 Volume & Density Distributions & Double Histograms

---

In [ ]:
with open(analysis_path+"density_voids.pk", 'rb') as f: density_voids = pkl.load(f)
with open(analysis_path+"dmedian_voids.pk", 'rb') as f: dmedian_voids = pkl.load(f)
with open(analysis_path+"volume_voids.pk",  'rb') as f: volume_voids  = pkl.load(f)

---

In [ ]:
sz_indx = 2

size    = BASELINES[sz_indx]["size"]; d_xyz = 75/size
R       = BASELINES[sz_indx]["R"]
lvl     = BASELINES[sz_indx]["lvl"]
cl      = BASELINES[sz_indx]["cl"]
MK      = BASELINES[sz_indx]["MK"]

---

In [ ]:
for i00, i000 in enumerate([2]):
    
    ud, od = ALL_uods[sz_indx][0][2][0][2][0][i000]
    path_to_plot = plots_path_0+"8.2___densities_volumes/"+str(ud)+"_"+str(od)+"/"
    if not os.path.exists(path_to_plot): os.makedirs(path_to_plot)

    titlez1 = [r"Volume distribution (all voids)"+"\n"
               +       "size="+str(size)
               + r"  |  $R=$"+str(R)+"cMpc/h"
               + r"  |  Lvl="+latex_float(lvl)
               + r"  |  "+cl
               +  "  |  "+MK
               + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]",
               "Density distribution (mean, all voids)"+"\n"
               +       "size="+str(size)
               + r"  |  $R=$"+str(R)+"cMpc/h"
               + r"  |  Lvl="+latex_float(lvl)
               + r"  |  "+cl
               +  "  |  "+MK
               + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]",
               "Density distribution (median, all voids)"+"\n"
               +       "size="+str(size)
               + r"  |  $R=$"+str(R)+"cMpc/h"
               + r"  |  Lvl="+latex_float(lvl)
               + r"  |  "+cl
               +  "  |  "+MK
               + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]"]
    
    xlbl1s = [r'Voids volume [(cMpc/h)$^3$]',
              r'Voids (mean) density [$\delta + 1$]',
              r'Voids (median) density [$\delta + 1$]']
    
    save_names1 = [path_to_plot+"HIST_voids_volumes.png",
                   path_to_plot+"HIST_voids_densities_mean.png",
                   path_to_plot+"HIST_voids_densities_median.png"]

    list_plots_all = [[], [], []]
    for i in range(len(ALL_Zs[sz_indx])):   # for the -2 walls
        list_plots_all[0].append(volume_voids[ i00][i][:-1])   
        list_plots_all[1].append(density_voids[i00][i][:-1])
        list_plots_all[2].append(dmedian_voids[i00][i][:-1])
    
    pixels = []; list_plots = []; bins_x = []; limits_y = []
    fin_pixel_val_y_max_list_all3 = []; fin_pixel_val_x_max_list_all3 = []; fin_pixel_ind_x_max_list_all3 = []; fin_pixel_going_max_list_all3 = []
    fin_pixel_val_y_min_list_all3 = []; fin_pixel_val_x_min_list_all3 = []; fin_pixel_ind_x_min_list_all3 = []; fin_pixel_going_min_list_all3 = []
    all_the_lists_all3 = [pixels, limits_y,
                          fin_pixel_val_y_max_list_all3, fin_pixel_val_x_max_list_all3, fin_pixel_ind_x_max_list_all3, fin_pixel_going_max_list_all3, 
                          fin_pixel_val_y_min_list_all3, fin_pixel_val_x_min_list_all3, fin_pixel_ind_x_min_list_all3, fin_pixel_going_min_list_all3,
                          list_plots, bins_x]

    for i0 in range(len(list_plots_all)):    
        for i00 in all_the_lists_all3: i00.append(0)
        pixels[i0], limits_y[i0], fin_pixel_val_y_max_list_all3[i0], fin_pixel_val_x_max_list_all3[i0], fin_pixel_ind_x_max_list_all3[i0], fin_pixel_going_max_list_all3[i0], fin_pixel_val_y_min_list_all3[i0], fin_pixel_val_x_min_list_all3[i0], fin_pixel_ind_x_min_list_all3[i0], fin_pixel_going_min_list_all3[i0], list_plots[i0], bins_x[i0] = hist_pixels_plot(list_plots_all[i0], np.min([np.min(list_plots_all[i0][_]) for _ in range(len(list_plots_all[i0]))]), np.max([np.max(list_plots_all[i0][_]) for _ in range(len(list_plots_all[i0]))]), titlez1[i0], save_names1[i0], ALL_Zs_float[sz_indx], xlbl1s[i0], int_if_possible=True, showoff=False)    



    titlez2 = [r"Colorcoded volume histogram (all voids)"+"\n"
               +       "size="+str(size)
               + r"  |  $R=$"+str(R)+"cMpc/h"
               + r"  |  Lvl="+latex_float(lvl)
               + r"  |  "+cl
               +  "  |  "+MK
               + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]",
               "Colorcoded density histogram (mean, all voids)"+"\n"
               +       "size="+str(size)
               + r"  |  $R=$"+str(R)+"cMpc/h"
               + r"  |  Lvl="+latex_float(lvl)
               + r"  |  "+cl
               +  "  |  "+MK
               + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]",
               "Colorcoded density histogram (median, all voids)"+"\n"
               +       "size="+str(size)
               + r"  |  $R=$"+str(R)+"cMpc/h"
               + r"  |  Lvl="+latex_float(lvl)
               + r"  |  "+cl
               +  "  |  "+MK
               + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]"]
    
    volume_density = ["volume",
                      "density",
                      "density"] 
    
    ylabels2 = [r'Voids volume [(cMpc/h)$^3$]',
                r'Voids (mean) density [$\delta + 1$]',
                r'Voids (median) density [$\delta + 1$]']
    
    save_names2 = [path_to_plot+"HIST_COL_voids_volumes.png",
                   path_to_plot+"HIST_COL_voids_densities_mean.png",
                   path_to_plot+"HIST_COL_voids_densities_median.png"]

    cbar_labels = ["Void counts normed", "Void counts normed", "Void counts normed"]
    
    for i0 in range(len(pixels)):
        color_double_hist_pixels_plot(pixels[i0], ALL_Zs_float[sz_indx], limits_y[i0], fin_pixel_val_y_max_list_all3[i0], fin_pixel_val_x_max_list_all3[i0], fin_pixel_ind_x_max_list_all3[i0],
                                      titlez2[i0], ylabels2[i0], save_names2[i0], cbar_labels[i0], volume_density[i0], pixels_bin_z, legend_loc=4, showoff=False)

---
---
---

## 8.3 Walls as a percentage of the grid

---

In [ ]:
sz_indx = 2

size    = BASELINES[sz_indx]["size"]; d_xyz = 75/size
nnc     = BASELINES[sz_indx]["nnc"]
R       = BASELINES[sz_indx]["R"]
lvl     = BASELINES[sz_indx]["lvl"]
cl      = BASELINES[sz_indx]["cl"]
uod     = BASELINES[sz_indx]["uod"]; ud, od = uod
MK      = BASELINES[sz_indx]["MK"]

file_path      = "../Modified_Data_"+str(size)+"/"

---

In [ ]:
walls_volume = []


for i00, i000 in enumerate([2]):
    
    ud, od = ALL_uods[sz_indx][0][2][0][2][0][i000]

    walls_volume.append([])
    
    for z_index in range(len(ALL_Zs[sz_indx])):

        Z = ALL_Zs[sz_indx][z_index]
    
        file_path_Z    = file_path     +"Z___"  +Z                   +"/"
        print(file_path_Z)
        file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
        file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
        file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"
        file_path_Duod = file_path_Lvl +"D___[" +str(ud)+"_"+str(od)+"]/"
        file_path_MK   = file_path_Duod+MK                           +"/"
    
        with open(file_path_MK+"fg___100.pk", 'rb') as f: fg_100 = pkl.load(f)
        # Re-order the void indices so they start from 0 and no gaps.
        fg_100[fg_100 != -2] = np.searchsorted(np.sort(np.unique(fg_100[fg_100 != -2])), fg_100[fg_100 != -2].ravel())
        fg_100_walls = fg_100 == -2
    
        walls_volume[-1].append(np.sum(fg_100_walls))

---

In [ ]:
with open(analysis_path+"walls_volume.pk", 'wb') as f: pkl.dump(walls_volume, f)

---

In [ ]:
with open(analysis_path+"walls_volume.pk", 'rb') as f: walls_volume = pkl.load(f)

In [ ]:
walls_volume = np.array(walls_volume)

---

In [ ]:
fig, axs = plt.subplots(figsize=(8,5), dpi=400)

fig.suptitle(r"Walls volume as a function of Z" + "\n"
             + "size=" + str(size)
             + r"  |  $R=$" + str(R) + "cMpc/h"
             + r"  |  Lvl=" + latex_float(lvl)
             + r"  |  " + cl
             +  "  |  " + MK
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")

x = np.arange(len(ALL_Zs[sz_indx]))

cols = ['#1f57b4', '#ff5f0e']
wls_max = [np.max(_)/size**3*100 for _ in walls_volume]
for i00, i000 in enumerate([2]):
    
    ud, od = ALL_uods[sz_indx][0][2][0][2][0][i000]
    
    # Add zorder to the bar plot to ensure it is drawn above the grid
    axs.bar(x, walls_volume[i00]/size**3*100, width=1, color=cols[i00], 
            edgecolor='black', linewidth=0.4, alpha=0.8, 
            label=r"$\Delta_{CDF}$=[" + str(ud) + "," + str(od) + "]", 
            zorder=3)  # Set zorder higher than grid
    
    axs.set_xticks(x, ALL_Zs_floatstr[sz_indx])
    tks, tkss = set_ticks(0, np.max(wls_max), log_lin=False, int_if_possible=True, clean=False, d_tks_notclean=1)
    for wls_max_i in wls_max:
        tks.append(wls_max_i)
        tkss.append(latex_float(wls_max_i, int_if_possible=True))
    axs.set_yticks(tks, tkss)
    
    axs.set_xlabel('Z')
    axs.set_ylabel('Volume (percent of the total grid)', labelpad=7)
    
    # Set grid zorder lower than bars
    axs.grid(which='major', axis='y', linestyle='-', alpha=0.9, zorder=0)
    axs.legend()

plt.tight_layout()
plt.savefig(plots_path_0 + "8.3___Walls_volume_percentage.png", bbox_inches='tight')
plt.close()

In [ ]:
fig, axs = plt.subplots(figsize=(8,3.5), dpi=400)

fig.suptitle(r"Walls volume as a function of Z" + "\n"
             + "size=" + str(size)
             + r"  |  $R=$" + str(R) + "cMpc/h"
             + r"  |  Lvl=" + latex_float(lvl)
             + r"  |  " + cl
             + "  |  " + MK
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")

x = np.arange(len(ALL_Zs[sz_indx]))

cols = ['#1f57b4', '#ff5f0e']
wls_max = [np.max(_)/size**3*100 for _ in walls_volume]
for i00, i000 in enumerate([2]):
    
    ud, od = ALL_uods[sz_indx][0][2][0][2][0][i000]
    
    # Add zorder to the bar plot to ensure it is drawn above the grid
    axs.plot(   x, walls_volume[i00]/size**3*100, color=cols[i00], linewidth=1.5, alpha=0.8,                                                            zorder=3)  # Set zorder higher than grid
    axs.scatter(x, walls_volume[i00]/size**3*100, color=cols[i00], s=5,         alpha=0.8, label=r"$\Delta_{CDF}$=[" + str(ud) + "," + str(od) + "]", zorder=3)  # Set zorder higher than grid
    
    axs.set_xticks(x, ALL_Zs_floatstr[sz_indx])
    tks, tkss = set_ticks(0, np.max(wls_max), log_lin=False, int_if_possible=True, clean=False, d_tks_notclean=1)
    for wls_max_i in wls_max:
        tks.append(wls_max_i)
        tkss.append(latex_float(wls_max_i, int_if_possible=True))
    axs.set_yticks(tks, tkss)
    
    axs.set_xlabel('Z')
    axs.set_ylabel('Volume (percent of the total grid)', labelpad=7)
    
    # Set grid zorder lower than bars
    #axs.grid(which='major', axis='y', linestyle='-', alpha=0.9, zorder=0)
    axs.legend()

    
#plt.ylim(0, 1.2*np.max(wls_max))
plt.grid()
plt.tight_layout()
plt.savefig(plots_path_0 + "8.3___Walls_volume_percentage_plot.png", bbox_inches='tight')
plt.close()

In [ ]:
del fg_100_walls; gc.collect()

---
---
---
---
---
---
---
---
---
---

# 9. Time-Derivative

---
---
---

## 9.1 Density Time Derivative Frames

---

In [ ]:
if not os.path.exists(plots_path_0+"9___Time_Derivative"): os.makedirs(plots_path_0+"9___Time_Derivative")

---

In [ ]:
sz_indx = 2

size    = BASELINES[sz_indx]["size"]; d_xyz = 75/size
nnc     = BASELINES[sz_indx]["nnc"]
R       = BASELINES[sz_indx]["R"]
lvl     = BASELINES[sz_indx]["lvl"]
cl      = BASELINES[sz_indx]["cl"]
uod     = BASELINES[sz_indx]["uod"]; ud, od = uod
MK      = BASELINES[sz_indx]["MK"]

Zs    = ALL_Zs[         sz_indx]
Zs_fs = ALL_Zs_floatstr[sz_indx]
ages  = ALL_ages[    sz_indx]

s_r = range(size)

file_path      = "../Modified_Data_"+str(size)+"/"

---

In [ ]:
cols = ['#006400', '#FFFF00', '#FF1493']
custom_cmap = mcolors.ListedColormap(cols)

In [ ]:
same_walls = True

In [ ]:
progress_bar(0, len(Zs))
for i0 in range(len(Zs)):
    
    i0 = len(Zs)-1-i0
    Z = Zs[i0]

    file_path_Z    = file_path     +"Z___"  +Z                   +"/"
    file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
    file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"
    file_path_Lvl  = file_path_R   +"Lvl___"+str(lvl)            +"/"+cl+"/"
    file_path_Duod = file_path_Lvl +"D___[" +str(ud)+"_"+str(od)+"]/"
    file_path_MK   = file_path_Duod+MK                           +"/"


    
    with open(file_path_R+"grid_fft_"+rho_delta+".pk", 'rb') as f: grid_fft = pkl.load(f)
    vmin_fft = np.min(grid_fft); vmax_fft = np.max(grid_fft)
    grid_fft_sc = grid_fft[sc]
    
    
    if i0 != len(Zs)-1:
        d_age = ages[i0+1]-ages[i0]
        grid_fft_sc_diff = (grid_fft_sc_before-grid_fft_sc)/d_age
    
    grid_fft_sc_before = copy.deepcopy(grid_fft_sc)
    
    
    
    with open(file_path_MK+"fg___100.pk", 'rb') as f: fg_100 = pkl.load(f)
    # Re-order the void indices so they start from 0 and no gaps.
    fg_100[fg_100 != -2] = np.searchsorted(np.sort(np.unique(fg_100[fg_100 != -2])), fg_100[fg_100 != -2].ravel())
    fg_100_walls = fg_100 == -2
    fg_100_sc    = fg_100[sc].tolist()

    walls_sc = []
    for i in s_r:
        walls_sc.append([])
        for j in s_r:
            if fg_100_sc[i][j] == -2: walls_sc[i].append(1)
            else:                     walls_sc[i].append(0)
    walls_sc = np.array(walls_sc)



   
    if i0 != len(Zs)-1:
        
        # for the left frame
        walls_sc_L = copy.deepcopy(walls_sc)
        walls_sc_L = np.ma.masked_where(walls_sc_L == 0, walls_sc_L)    # where there is no wall
        
        # for the right frame
        if same_walls:
            walls_sc_R1 = copy.deepcopy(walls_sc)
        else:
            walls_sc_R1 = 2*walls_sc + walls_sc_before
        walls_sc_R = np.ma.masked_where(walls_sc_R1 == 0, walls_sc_R1)    # where there is no wall
        
        # background under the walls
        grid_fft_sc_under = np.zeros((size, size))
        for i in range(size):
            for j in range(size):
                if walls_sc_R1[i][j] > 0:
                    grid_fft_sc_under[i][j] = grid_fft_sc_diff[i][j]
        
        grid_fft_sc_under_positive = np.ma.masked_where(grid_fft_sc_under <  0, grid_fft_sc_under)
        grid_fft_sc_under_negative = np.ma.masked_where(grid_fft_sc_under >= 0, grid_fft_sc_under)
        
        
        X, Y  = np.mgrid[75:0:complex(0, size), 0:75:complex(0, size)]
        
        ZZ    = np.ma.masked_array(grid_fft_sc_diff, walls_sc_R > 0)
        
    walls_sc_before = copy.deepcopy(walls_sc)
    
    
    
    
    
    ########
    # PLOT
    
    
    if i0 != len(Zs)-1:
        
        fig, axs = plt.subplots(1, 2, figsize=(14,6), dpi=400)
        fig.suptitle(  r"Density and Density Time-Derivative maps" + "\n"
                     + r"$yz$-slice [cMpc/h]"
                     +  "  |  size="+str(size)
                     + r"  |  $R=$"+str(R)+"cMpc/h"
                     + r"  |  Lvl="+latex_float(lvl)
                     + r"  |  "+cl
                     +  "  |  "+MK
                     + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")
        
        for i2 in range(2):
            divider = make_axes_locatable(axs[i2])
            cax = divider.append_axes('right', size='5%', pad=0.05)
        
        
            
            if i2 == 0:
                axs[i2].set_title('z='+Zs_fs[i0])
                
                vmin1 = np.min(grid_fft_sc); vmax1 = np.max(grid_fft_sc)
        
                imm = axs[i2].imshow(grid_fft_sc, norm=LogNorm(vmin=vmin_fft, vmax=vmax_fft), cmap='inferno', extent=[0.0, 75, 0.0, 75])
                
                cbar = plt.colorbar(imm, cax=cax, orientation='vertical')
                tks, tkss = set_ticks(vmin_fft, vmax_fft, factor_cut=0.1, ignore_tks_in_tkss=False)
                cbar.ax.set_yticks(tks, tkss)
                cbar.set_label(r'$\delta + 1$', rotation=270, labelpad=15)
        
                imm = axs[i2].imshow(walls_sc_L, cmap=custom_cmap, extent=[0.0, 75, 0.0, 75], alpha=1)
        
        
        
            else:
                axs[i2].set_title(r"$\Delta t =$" + str(round(ages[i0]-ages[i0+1],3))+'Gyr')
                
                vmin2 = np.min(ZZ); vmax2 = np.max(ZZ)
        
                imm = axs[i2].pcolormesh(Y, X, ZZ, norm=colors.SymLogNorm(linthresh=0.03, linscale=0.03, vmin=vmin2, vmax=vmax2), cmap='RdBu_r', alpha=1)
        
                cbar = plt.colorbar(imm, cax=cax, ax=axs[i2])
                tks, tkss = set_ticks(vmin2, vmax2, log_lin=True, factor_cut=0.1, ignore_tks_in_tkss=False)
                for i00, label in enumerate(tkss):
                    if   label == r'$-10^{-2}$': tkss[i00] = ' '
                    elif label == r'$10^{-2}$':  tkss[i00] = ' '
                cbar.ax.set_yticks(tks, tkss)
                cbar.set_label(r'$\Delta \delta / \Delta t$ [Gyr$^{-1}$]', rotation=270, labelpad=15)
        
                #imm = axs[i2].imshow(grid_fft_sc_under_positive, norm=colors.SymLogNorm(linthresh=0.03, linscale=0.03, vmin=vmin2, vmax=vmax2), cmap='RdBu_r', extent=[0.0, 75, 0.0, 75])
                #imm = axs[i2].imshow(grid_fft_sc_under_negative, norm=colors.SymLogNorm(linthresh=0.03, linscale=0.03, vmin=vmin2, vmax=vmax2), cmap='RdBu_r', extent=[0.0, 75, 0.0, 75])
                
                imm = axs[i2].imshow(walls_sc_R, cmap=custom_cmap, extent=[0.0, 75, 0.0, 75], alpha=1)
                
                if not same_walls:
                    textstr = '\n'.join(("pink    - overlap",
                                         "yellow - new walls (left frame)",
                                         "green  - older walls"))
                    props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
                    axs[i2].text(0.05, 0.95, textstr, transform=axs[i2].transAxes, fontsize=10, verticalalignment='top', bbox=props)
        
        
            axs[i2].set_xticks(range_75_5)
            axs[i2].set_yticks(range_75_5)
        
            ccc = divider.append_axes('right', size='25%', pad=0.1)
            ccc.set_xticks([]); ccc.set_xticklabels([])
            ccc.set_yticks([]); ccc.set_yticklabels([])
            ccc.axis('off')
        
        
        
        plt.tight_layout()
        plt.savefig(plots_path_0+"9___Time_Derivative/Frame__"+str(Z)+".png", bbox_inches='tight')
        plt.close()
    
    
    progress_bar(len(Zs)-i0, len(Zs))

---

In [ ]:
INPUT_FOLDER = Path(plots_path_0+"9___Time_Derivative/")

PREFIX    = None
EXTENSION = ".png"

OUTPUT_NAME   = "Time_derivative_video.mp4"
OUTPUT_FOLDER = None   # None = same as INPUT_FOLDER

DISPLAY_SECONDS = 1
VIDEO_FPS       = int(1/DISPLAY_SECONDS)   # I mean, you could always use sth else...

CODEC = "avc1"   # I can see this one with QuickTime Player but feel free to change it


make_video(reverse_ORDER = True)

---

In [ ]:
del fg_100_walls, fg_100_sc, grid_fft_sc_before, walls_sc, walls_sc_before, grid_fft_sc_diff, walls_sc_R1, grid_fft_sc_under; gc.collect()

---
---
---

## 9.2 Density Time Derivative Histogram

---

In [ ]:
sz_indx = 2

size    = BASELINES[sz_indx]["size"]; d_xyz = 75/size
nnc     = BASELINES[sz_indx]["nnc"]
R       = BASELINES[sz_indx]["R"]
lvl     = BASELINES[sz_indx]["lvl"]
cl      = BASELINES[sz_indx]["cl"]
uod     = BASELINES[sz_indx]["uod"]; ud, od = uod
MK      = BASELINES[sz_indx]["MK"]

Zs    = ALL_Zs[      sz_indx]
Zs_f  = ALL_Zs_float[sz_indx]
ages  = ALL_ages[    sz_indx]

file_path      = "../Modified_Data_"+str(size)+"/"

---

In [ ]:
no_layers = 128   # for the full 512 or even 256 on the x axis (i.e. using the whole cube, my 16Gb RAM blows-up.... c'est la vie)

In [ ]:
GRAD = []

for i0 in range(len(Zs)):

    Z = Zs[i0]

    file_path_Z    = file_path     +"Z___"  +Z                   +"/"
    file_path_NNC  = file_path_Z   +"NNC___"+str(nnc)            +"/"
    file_path_R    = file_path_NNC +"R___"  +str(R)              +"/"


    with open(file_path_R+"grid_fft_delta.pk", 'rb') as f: grid_fft = pkl.load(f)
    
    if i0 != 0:
        d_age = ages[i0]-ages[i0-1]
        GRAD.append((grid_fft[0:no_layers]-grid_fft_before[0:no_layers])/d_age)
    
    grid_fft_before = copy.deepcopy(grid_fft)

In [ ]:
list_plots_all = [[i.flatten().tolist() for i in GRAD]]

In [ ]:
del GRAD; gc.collect()

---

In [ ]:
lmin1s = [     -10**2]
lmax1s = [      10**2]
zero_rounds = [ 10**-3]

tits = [r"Density Time-Derivative Distribution (all voids)"+"\n"
       +       "size="+str(size)
       + r"  |  $R=$"+str(R)+"cMpc/h"
       + r"  |  Lvl="+latex_float(lvl)
       + r"  |  "+cl
       +  "  |  "+MK
       + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]"]

xlbl1s = [r'$\Delta \delta / \Delta t$ [Gyr$^{-1}$]']
ylbl1s = [r'Percentage of Cube at Set Density Time Derivative [%]']

save_names = [plots_path_0+"9___Time_Derivative/HIST_voids_densities_time_deriv.png"]

In [ ]:
len_list_plot_all = []; pixels = []; list_plots = []; bins_x = []

fin_pixel_val_y_max_list = []; fin_pixel_val_x_max_list = []; fin_pixel_ind_x_max_list = []
fin_pixel_val_y_min_list = []; fin_pixel_val_x_min_list = []; fin_pixel_ind_x_min_list = []
fin_pixel_going_max_list = []; fin_pixel_going_min_list = []

going = []; plt_lims = []; limits_y = []

all_the_lists = [len_list_plot_all,
                 pixels,
                 limits_y,
                 fin_pixel_val_y_max_list, fin_pixel_val_x_max_list, fin_pixel_ind_x_max_list, fin_pixel_going_max_list,
                 fin_pixel_val_y_min_list, fin_pixel_val_x_min_list, fin_pixel_ind_x_min_list, fin_pixel_going_min_list,
                 list_plots, bins_x,
                 going, plt_lims]

---

In [ ]:
legend_labels_Z = np.array(Zs_f[:-1]) #(np.array(Zs_f[1:]) + np.array(Zs_f[:-1]))/2

In [ ]:
for i0 in range(len(list_plots_all)):
    for i00 in all_the_lists: 
        i00.append(0)
    len_list_plot_all[i0], pixels[i0], limits_y[i0], fin_pixel_val_y_max_list[i0], fin_pixel_val_x_max_list[i0], fin_pixel_ind_x_max_list[i0], fin_pixel_val_y_min_list[i0], fin_pixel_val_x_min_list[i0], fin_pixel_ind_x_min_list[i0], list_plots[i0], bins_x[i0], going[i0], plt_lims[i0] = hist_pixels_plot(list_plots_all[i0], lmin1s[i0], lmax1s[i0], tits[i0], save_names[i0], legend_labels_Z, xlbl1s[i0], ylbl1s[i0], tks_scale_y=1/size**2/no_layers * 100, zero_round=zero_rounds[i0], get_a_stable_exponent_y=True, int_if_possible=True, showoff=False, all3=False, return_lengths=True)

In [ ]:
with open(analysis_path+"pixels.pk",                   'wb') as f: pkl.dump(pixels,                   f)
with open(analysis_path+"limits_y.pk",                 'wb') as f: pkl.dump(limits_y,                 f)
with open(analysis_path+"fin_pixel_val_y_max_list.pk", 'wb') as f: pkl.dump(fin_pixel_val_y_max_list, f)
with open(analysis_path+"fin_pixel_val_x_max_list.pk", 'wb') as f: pkl.dump(fin_pixel_val_x_max_list, f)
with open(analysis_path+"fin_pixel_ind_x_max_list.pk", 'wb') as f: pkl.dump(fin_pixel_ind_x_max_list, f)

---
---
---

In [ ]:
for i0, i in enumerate(len_list_plot_all):
    for j0, j in enumerate(i):
        j = np.array(j)
        j = j/size**2/no_layers * 100
        len_list_plot_all[i0][j0] = j.tolist()

---

In [ ]:
zPaths_no_mean = np.array(Zs_f[:-1])

In [ ]:
fig, axs = plt.subplots(figsize=(10,7), dpi=400)

fig.suptitle(  r"Evolution of the Sign of the Density Time Derivative" + "\n"
             +       "size="+str(size)
             + r"  |  $R=$"+str(R)+"cMpc/h"
             + r"  |  Lvl="+latex_float(lvl)
             + r"  |  "+cl
             +  "  |  "+MK
             + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]")


axs.plot(   zPaths_no_mean, len_list_plot_all[0][0], c='r', lw=1, alpha=0.6)
axs.scatter(zPaths_no_mean, len_list_plot_all[0][0], c='r', s =7, label="Negative derivative")

axs.plot(   zPaths_no_mean, len_list_plot_all[0][1], c='b', lw=1, alpha=0.6)
axs.scatter(zPaths_no_mean, len_list_plot_all[0][1], c='b', s =7, label="Null         derivative")

axs.plot(   zPaths_no_mean, len_list_plot_all[0][2], c='g', lw=1, alpha=0.6)
axs.scatter(zPaths_no_mean, len_list_plot_all[0][2], c='g', s =7, label="Positive   derivative")


axs.set_xlim(-0.5,zPaths_no_mean[-1]+0.5)

xtks = [zPaths_no_mean[0], zPaths_no_mean[4]]
for i in zPaths_no_mean[6:]: xtks.append(i)

axs.set_xticks(xtks)

axs.set_ylabel("Percentage of the Data Set [%]")
axs.set_xlabel("Z mean (between the two frames)")

axs.legend(loc=1)
axs.grid()

plt.tight_layout()
plt.savefig(plots_path_0+"9___Time_Derivative/Sign_Evolution.png", bbox_inches='tight')
plt.close()

---

In [ ]:
tits = ['Colorcoded Density Time-Derivative Histogram' + "\n"
        +  "size="+str(size)
        + r"  |  $R=$"+str(R)+"cMpc/h"
        + r"  |  Lvl="+latex_float(lvl)
        + r"  |  "+cl
        +  "  |  "+MK
        + r"  |  $\Delta_{CDF}$=["+str(ud)+","+str(od)+"]"]

ylabels = [r'Density Time-Derivative $\Delta \delta / \Delta t$ [Gyr$^{-1}$]']

volume_density = ['density time-derivative']

save_names = [plots_path_0+"9___Time_Derivative/HIST_COL_voids_densities_time_deriv.png"]

cbar_labels = ['Percentage of grid at set density time-derivative [%]']

---

In [ ]:
with open(analysis_path+"pixels.pk",                   'rb') as f: pixels                   = pkl.load(f)
with open(analysis_path+"limits_y.pk",                 'rb') as f: limits_y                 = pkl.load(f)
with open(analysis_path+"fin_pixel_val_y_max_list.pk", 'rb') as f: fin_pixel_val_y_max_list = pkl.load(f)
with open(analysis_path+"fin_pixel_val_x_max_list.pk", 'rb') as f: fin_pixel_val_x_max_list = pkl.load(f)
with open(analysis_path+"fin_pixel_ind_x_max_list.pk", 'rb') as f: fin_pixel_ind_x_max_list = pkl.load(f)

zPaths_no_mean = np.array(Zs_f[:-1])
zero_rounds = [ 10**-3]

---

In [ ]:
for i0 in range(len(pixels)):
    color_double_hist_pixels_plot(pixels[i0], [round(_,2) for _ in zPaths_no_mean], limits_y[i0], fin_pixel_val_y_max_list[i0], fin_pixel_val_x_max_list[i0], fin_pixel_ind_x_max_list[i0], tits[i0], ylabels[i0], save_names[i0], cbar_labels[i0], volume_density[i0], zero_rounds[i0], pixels_bin_z, showoff=False, int_if_possible=False, print_all3_lines=True, prend=False, plot_counts=False)

---

In [ ]:
del grid_fft_before, all_the_lists, list_plots, list_plots_all; gc.collect()

---
---
---